# Pipeline OCR/VLM V8 — Domiciliation des salaires étrangers

**Architecture : cascade de modèles + vLLM + hybride texte natif**

Évolutions majeures par rapport à la V7.1 (objectif : diviser le temps et les
tokens par 3 à 5, sans perte de qualité — enjeu réglementaire) :

1. **Cascade de modèles.** Qwen2.5-VL-7B traite en premier passage toutes les
   pages dactylographiées (même scannées : scan net = tâche facile). Le modèle
   lourd (Qwen3.6-35B-A3B-FP8, MoE ~3B paramètres actifs, décodage 2-3x plus
   rapide qu'un 27B dense) n'est chargé qu'ensuite, pour :
   - la planche permis de travail (scan dégradé, bilingue, cachets) ;
   - l'escalade des pages dont le taux de remplissage reste < 0.70.
   Les deux moteurs sont chargés séquentiellement (pas de conflit NCCL).

2. **vLLM offline** : continuous batching, prefix caching (les prompts
   d'extraction identiques ne sont pré-encodés qu'une fois), FP8 natif
   (plus de déquantification bf16), CUDA graphs. Résout aussi l'absence de
   flash-attn (backends attention internes vLLM, efficaces sur H100).

3. **Régime token.** Classification : prompt court (~100 tokens), sortie 60.
   Extraction dactylographiée : image ~1150 px, sortie max 500. Planche permis
   : HD recadrée, sortie max 700. Budget cible : ~14k tokens/PDF au lieu de 40k+.

4. **Hybride texte natif.** Si le PDF possède une couche texte (> 200 car.),
   la classification est faite par mots-clés et l'extraction par regex sur les
   libellés : coût nul, zéro hallucination. Ne concerne qu'une minorité de
   dossiers (la plupart sont des scans) mais c'est gratuit et auditable.

5. **Batching transverse.** Classification et extraction sont regroupées par
   type de document : toutes les pages d'un PDF passent ensemble, puis toutes
   les pages de tous les PDFs du lot dans la vague de passe 1.

Le pipeline ne réalise toujours pas les contrôles réglementaires finaux
(Alteryx conserve les règles métier et les décisions).

**Prérequis Domino :** `pip install -U vllm` (une fois, dans l'environnement ;
vLLM apporte ses propres kernels attention — flash-attn non requis sur H100).


In [ ]:
# =====================================================================
# 1. Imports et vérification d'environnement
# =====================================================================
import base64
import gc
import hashlib
import io
import json
import re
import sys
import time
import calendar
import psutil
from collections import defaultdict
from datetime import date, datetime, timedelta
from pathlib import Path

import fitz
import numpy as np
import pandas as pd
import torch
from PIL import Image
from openpyxl import Workbook
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter

try:
    from vllm import LLM, SamplingParams
except ImportError as exc:
    raise RuntimeError(
        "vLLM est requis en V8. Installer une fois dans l'environnement Domino : "
        "%pip install -U vllm  (puis redémarrer le kernel)."
    ) from exc

print("Python :", sys.version.split()[0])
print("Torch  :", torch.__version__, "| CUDA :", torch.cuda.is_available(),
      "| GPU :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")
print("vLLM   :", __import__("vllm").__version__)
print("PyMuPDF:", fitz.__doc__.splitlines()[0] if fitz.__doc__ else "OK")
print("✅ Imports OK")


In [ ]:
# =====================================================================
# 2. Configuration V8
# =====================================================================
# --- Modèles (adapter les chemins au ModelHub Domino) ----------------
MODELE_LEGER_PATH = "/domino/edv/modelhub/ModelHub-model-huggingface-Qwen/Qwen2.5-VL-7B-Instruct/main"
MODELE_LOURD_PATH = "/domino/edv/modelhub/ModelHub-model-huggingface-Qwen/Qwen3.6-35B-A3B-FP8/main"

PIPELINE_VERSION = "DOM_V8_CASCADE_VLLM"

# --- Rendu adaptatif -------------------------------------------------
PDF_ZOOM = 2.2                      # pages dactylographiées (scan net suffit)
IMAGE_MAX_SIZE = 1150               # ~1500 tokens image/page au lieu de ~2500
PDF_ZOOM_HAUTE_DEF = 4.5            # planche permis
IMAGE_MAX_SIZE_HAUTE_DEF = 2000

# --- Budget tokens génération ---------------------------------------
MAX_NEW_TOKENS_CLASSIFICATION = 60  # un JSON de 2 clés
MAX_NEW_TOKENS_EXTRACTION_TYPED = 500    # ~30 champs ≈ 350 tokens
MAX_NEW_TOKENS_EXTRACTION_PERMIS = 700   # planche permis plus dense
MAX_NUM_SEQS = 32                   # profondeur du continuous batching

# --- Escalade ---------------------------------------------------------
SEUIL_REMPLISSAGE_OK = 0.70
FRONTIERE_ZONE_RECHERCHE = (0.28, 0.66)
FRONTIERE_SEUIL_ENCRE = 0.004
FRONTIERE_DEFAUT = 0.47

# --- Entrées / sorties ------------------------------------------------
INPUT_DIR  = Path('/mnt/data/domiciliations_in')
OUTPUT_DIR = Path('/mnt/data/domiciliations_out')
JSON_DIR   = OUTPUT_DIR / 'json_dossiers'
LOG_PATH   = OUTPUT_DIR / 'pipeline_domiciliations.log'
EXCEL_PATH = OUTPUT_DIR / f"domiciliations_{datetime.now().strftime('%Y%m%d_%H%M')}.xlsx"
MASTER_JSON_PATH = OUTPUT_DIR / 'domiciliations_master.json'
PRORATA_MODE = 'CALENDAR_DAYS'
DOM_REFERENCE_EXCEL = Path("/mnt/data/fichier_domiciliations.xlsx")
PREDOM_REFERENCE_EXCEL = Path("/mnt/data/fichier_predomiciliations.xlsx")
EXCEL_AMOUNT_FORMAT = "0.00"

for d in (INPUT_DIR, OUTPUT_DIR, JSON_DIR):
    d.mkdir(parents=True, exist_ok=True)

pdfs = sorted(INPUT_DIR.glob('*.pdf'))
print(f'PDFs détectés : {len(pdfs)} | Entrée : {INPUT_DIR} | Sortie : {OUTPUT_DIR}')
print('Léger :', MODELE_LEGER_PATH)
print('Lourd :', MODELE_LOURD_PATH)


In [ ]:
# =====================================================================
# 3. Utilitaires PDF, image, log, JSON
# =====================================================================
def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

def resize_image(img, max_side=IMAGE_MAX_SIZE):
    w, h = img.size
    if max(w, h) <= max_side:
        return img
    ratio = max_side / max(w, h)
    return img.resize((int(w * ratio), int(h * ratio)), Image.LANCZOS)

def white_ratio(image):
    arr = np.array(image.convert("L"))
    return float((arr > 245).sum() / arr.size)

def is_blank(image, threshold=0.995):
    return white_ratio(image) >= threshold

def pdf_to_pages(path, zoom=PDF_ZOOM):
    path = Path(path)
    if not path.exists() or path.stat().st_size == 0:
        raise ValueError(f"PDF introuvable ou vide : {path}")
    pages = []
    doc = fitz.open(str(path))
    try:
        matrix = fitz.Matrix(zoom, zoom)
        for i in range(int(doc.page_count)):
            pix = doc.load_page(i).get_pixmap(matrix=matrix, alpha=False)
            if pix.width <= 0 or pix.height <= 0 or not pix.samples:
                raise ValueError(f"Rendu vide : {path.name}, page {i + 1}")
            img = Image.frombytes("RGB", (pix.width, pix.height), pix.samples)
            img = resize_image(img)
            pages.append({
                "index": i, "page_num": i + 1, "image": img,
                "width": img.width, "height": img.height,
                "white_ratio": round(white_ratio(img), 6),
            })
    finally:
        doc.close()
    return pages

def render_page_region(pdf_path, page_index, zoom=PDF_ZOOM_HAUTE_DEF,
                       max_side=IMAGE_MAX_SIZE_HAUTE_DEF, crop=None):
    pdf_path = Path(pdf_path)
    doc = fitz.open(str(pdf_path))
    try:
        pix = doc.load_page(int(page_index)).get_pixmap(matrix=fitz.Matrix(zoom, zoom), alpha=False)
        img = Image.frombytes("RGB", (pix.width, pix.height), pix.samples)
    finally:
        doc.close()
    if crop:
        img = crop_region(img, *crop)
    return resize_image(img, max_side=max_side)

def crop_region(image, haut=0.0, bas=1.0, gauche=0.0, droite=1.0):
    largeur, hauteur = image.size
    x0 = int(max(0.0, min(1.0, gauche)) * largeur); x1 = int(max(0.0, min(1.0, droite)) * largeur)
    y0 = int(max(0.0, min(1.0, haut)) * hauteur);  y1 = int(max(0.0, min(1.0, bas)) * hauteur)
    if x1 <= x0 or y1 <= y0:
        return image
    return image.crop((x0, y0, x1, y1))

def detecter_frontiere_documents(image, zone=FRONTIERE_ZONE_RECHERCHE,
                                 seuil_encre=FRONTIERE_SEUIL_ENCRE, defaut=FRONTIERE_DEFAUT):
    try:
        arr = np.array(image.convert("L"))
    except Exception:
        return defaut
    hauteur = arr.shape[0]
    if hauteur < 10:
        return defaut
    densite = (arr < 200).sum(axis=1) / max(arr.shape[1], 1)
    y_min, y_max = int(zone[0] * hauteur), int(zone[1] * hauteur)
    bandes, debut = [], None
    for y in range(y_min, max(y_max, y_min + 1)):
        vide = densite[y] < seuil_encre
        if vide and debut is None:
            debut = y
        elif not vide and debut is not None:
            bandes.append((debut, y)); debut = None
    if debut is not None:
        bandes.append((debut, y_max))
    if not bandes:
        return defaut
    d, f = max(bandes, key=lambda b: b[1] - b[0])
    if (f - d) / hauteur < 0.015:
        return defaut
    return round((d + f) / 2 / hauteur, 4)

def crops_planche_permis(image, marge=0.02):
    frontiere = detecter_frontiere_documents(image)
    bas_titre = min(1.0, frontiere + marge)
    haut_couverture = max(0.0, frontiere - marge)
    return {
        "frontiere": frontiere,
        "titre": (0.00, bas_titre, 0.00, 1.00),
        "colonne_identite": (0.00, bas_titre, 0.44, 1.00),
        "colonne_poste": (0.00, bas_titre, 0.00, 0.56),
        "couverture": (haut_couverture, 1.00, 0.00, 1.00),
    }

def parse_json_response(text):
    if not text:
        return {}
    clean = re.sub(r"^```(?:json)?", "", str(text).strip(), flags=re.I).strip()
    clean = re.sub(r"```$", "", clean).strip()
    m = re.search(r"\{.*\}", clean, flags=re.S)
    if not m:
        return {}
    candidate = m.group(0)
    for attempt in (candidate, re.sub(r",\s*([}\]])", r"\1", candidate)):
        try:
            parsed = json.loads(attempt)
            return parsed if isinstance(parsed, dict) else {}
        except Exception:
            continue
    return {}

def log(message):
    line = f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - {message}"
    print(line)
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(line + "\n")

def estimate_image_tokens(img):
    # Qwen-VL : patch 14 px fusionnés 2x2 -> ~1 token par bloc 28x28
    return int(np.ceil(img.width / 28) * np.ceil(img.height / 28))

def taux_remplissage(data, champs_attendus):
    if not champs_attendus:
        return 1.0
    remplis = sum(1 for c in champs_attendus if (data or {}).get(c) not in (None, ""))
    return round(remplis / len(champs_attendus), 4)

print("✅ Utilitaires PDF/image/JSON OK")


In [ ]:
# =====================================================================
# 4. Hybride texte natif : classification par mots-clés + regex
# =====================================================================
# Ne s'applique qu'aux PDF "nés numériques" (couche texte). Coût nul,
# zéro hallucination : les libellés sont ceux des prompts VLM.
# Ordre important : CONTRAT_SPECIFIQUE avant CONTRAT_TRAVAIL (son titre
# contient "CONTRAT DE TRAVAIL ... SPECIFIQUE ...").

def native_keyword_classify(text):
    t = re.sub(r"\s+", " ", text.upper())
    if "ENGAGEMENT DE DOMICILIATION" in t or "CONTRAT DES SALARIES ETRANGERS" in t:
        return "ENGAGEMENT_DOMICILIATION"
    if "MAIN D'OEUVRE ETRANGERE" in t or "MAIN D OEUVRE ETRANGERE" in t:
        return "CONTRAT_SPECIFIQUE"
    if "CONTRAT DE TRAVAIL A DUREE DETERMINEE" in t:
        return "CONTRAT_TRAVAIL"
    if ("DATE DE NAISSANCE" in t and "NATIONALIT" in t) or \
       ("PERMIS DE TRAVAIL" in t and "NOM DE L'ORGANISME EMPLOYEUR" in t):
        return "TITRE_TRAVAIL"
    if "PERMIS DE TRAVAIL" in t and "LOI" in t:
        return "PERMIS_TRAVAIL_COUVERTURE"
    return None

def _rx(pattern, text, group=1, flags=re.I):
    m = re.search(pattern, text, flags)
    if not m:
        return None
    val = m.group(group).strip()
    val = re.sub(r"[.\s]+$", "", val).strip()
    return val or None

def _clean_lines(text):
    return re.sub(r"[ \t]+", " ", text)

NATIVE_PATTERNS = {
    "ENGAGEMENT_DOMICILIATION": {
        "DOM_NOM_RAISON_SOCIAL_CLIENT": r"Nom et raison sociale\s*:?\s*(.+)",
        "DOM_COMPTE_LOCAL": r"N[°o]?\s*de compte\s*:?\s*([0-9][0-9\s]*)",
        "DOM_ADRESSE_CLIENT": r"Adresse\s*:?\s*(.+)",
        "DOM_AGENCE_DOMICILIATAIRE": r"Agence domiciliataire\s*:?\s*(.+)",
        "DOM_NUMERO_CONTRAT": r"Num(?:é|e)ro du contrat\s*:?\s*(.+)",
        "DOM_DUREE_CONTRAT_MOIS": r"Dur(?:é|e)e du contrat\s*:?\s*(.+)",
        "DOM_DATE_DEBUT_CONTRAT": r"Date de d(?:é|e)but de contrat\s*:?\s*(\d{2}/\d{2}/\d{4})",
        "DOM_DATE_FIN_CONTRAT": r"Date de fin de contrat\s*:?\s*(\d{2}/\d{2}/\d{4})",
        "DOM_NOM_RAISON_SOCIAL_EMPLOYEUR": r"Nom et raison sociale de L'Employeur\s*:?\s*(.+)",
        "DOM_ADRESSE_EMPLOYEUR": r"Adresse de L'Employeur\s*:?\s*(.+)",
        "DOM_SALAIRE_NET_MENSUEL": r"Salaire net mensuel\s*:?\s*([0-9][0-9\s.,]*)",
        "DOM_PART_TRANSFERABLE": r"Montant de la part transf(?:é|e)rable\s*:?\s*([0-9][0-9\s.,]*)",
        "DOM_TAUX_TRANSFERABLE": r"Pourcentage en regard du salaire net mensuel\s*:?\s*([0-9]+\s*%)",
        "DOM_MONTANT_TOTAL_DOMICILIE": r"Montant domicili(?:é|e) en DZD\s*:?\s*([0-9][0-9\s.,]*)",
    },
    "CONTRAT_TRAVAIL": {
        "CTR_REFERENCE_DOCUMENT": r"\b(TR-\d+)\b",
        "CTR_TYPE": None,  # fixé en dur ci-dessous
        "CTR_EMPLOYEUR": r"employeur ci-apr(?:è|e)s d(?:é|e)sign(?:é|e)\s*:\s*(.+)",
        "CTR_ACTIVITE_EMPLOYEUR": r"Nature de l'activit(?:é|e)\s*:?\s*(.+)",
        "CTR_DUREE_MOIS": r"dur(?:é|e)e de\s*:?\s*(\d+)\s*Mois",
        "CTR_DATE_DEBUT_CONTRAT": r"(?:à compter du|a compter du)\s*:?\s*(\d{2}/\d{2}/\d{4})",
        "CTR_POSTE": r"En qualit(?:é|e) de\s*:?\s*(.+)",
        "CTR_NOM_PRENOM_TRAVAILLEUR": r"A \(Mr/Mme\)\s*:?\s*(.+)",
        "CTR_PERE_NOM_PRENOM": r"Fils de\s*:?\s*(.+?)\s+et de\s*:",
        "CTR_MERE_NOM_PRENOM": r"et de\s*:\s*(.+)",
        "CTR_NATIONALITE": r"Nationalit(?:é|e)\s*:?\s*(.+)",
        "CTR_DATE_NAISSANCE": r"N(?:é|e)\(e\) le\s*:?\s*(\d{2}/\d{2}/\d{4})",
        "CTR_LIEU_PAYS_NAISSANCE": r"N(?:é|e)\(e\) le\s*:?\s*\d{2}/\d{2}/\d{4}\s*(?:à|:)\s*(.+)",
        "CTR_ADRESSE_ALGERIE": r"Adresse en Alg(?:é|e)rie\s*:?\s*(.+)",
        "CTR_QUALIFICATION": r"Qualification professionnelle\s*:?\s*(.+)",
        "CTR_NUMERO_PERMIS_TRAVAIL": r"permis de travail N(?:°|o)\s*([0-9\-/]+)",
        "CTR_DATE_DELIVRANCE_PERMIS": r"D(?:é|e)livr(?:é|e) le\s*:?\s*(\d{2}/\d{2}/\d{4})",
        "CTR_DATE_DEBUT_VALIDITE_PERMIS": r"Valable du\s*(\d{2}/\d{2}/\d{4})",
        "CTR_DATE_FIN_VALIDITE_PERMIS": r"Valable du\s*\d{2}/\d{2}/\d{4}\s*au\s*(\d{2}/\d{2}/\d{4})",
        "CTR_SALAIRE_BRUT": r"salaire mensuel brut\s*:?\s*([0-9][0-9\s.,]*)",
        "CTR_SALAIRE_NET": r"salaire mensuel net\s*:?\s*([0-9][0-9\s.,]*)",
        "CTR_AFFILIATION_SS": r"s(?:é|e)curit(?:é|e) sociale\s*:?\s*(.+)",
        "CTR_NUMERO_EMPLOYEUR": r"Employeur\s*:?\s*(\d{6,})",
        "CTR_DATE_SIGNATURE": r"Fait à\s*:?\s*.+?le\s*(\d{2}/\d{2}/\d{4})",
    },
    "CONTRAT_SPECIFIQUE": {
        "CTS_REFERENCE_DOCUMENT": r"\b(TR-\d+)\b",
        "CTS_SAP_ID": r"SAP id\s*-\s*(\d+)",
        "CTS_EMPLOYEUR": r"employeur ci-apr(?:è|e)s d(?:é|e)sign(?:é|e)\s*:\s*(.+)",
        "CTS_ACTIVITE_EMPLOYEUR": r"Nature de l'activit(?:é|e)\s*:?\s*(.+)",
        "CTS_DUREE_MOIS": r"dur(?:é|e)e de\s*:?\s*(\d+)\s*Mois",
        "CTS_DATE_DEBUT_CONTRAT": r"(?:à compter du|a compter du|A compter du)\s*:?\s*(\d{2}/\d{2}/\d{4})",
        "CTS_POSTE": r"en qualit(?:é|e) de\s*:?\s*(.+)",
        "CTS_NOM_PRENOM_TRAVAILLEUR": r"A \(Mr/Mme\)\s*:?\s*(.+)",
        "CTS_PERE_NOM_PRENOM": r"Fils de\s*:?\s*(.+?)\s+et de\s*:",
        "CTS_MERE_NOM_PRENOM": r"et de\s*:\s*(.+)",
        "CTS_NATIONALITE": r"Nationalit(?:é|e)\s*:?\s*(.+)",
        "CTS_DATE_NAISSANCE": r"N(?:é|e)\(e\) le\s*:?\s*(\d{2}/\d{2}/\d{4})",
        "CTS_LIEU_PAYS_NAISSANCE": r"N(?:é|e)\(e\) le\s*:?\s*\d{2}/\d{2}/\d{4}\s*(?:à|:)\s*(.+)",
        "CTS_ADRESSE_ALGERIE": r"Adresse en Alg(?:é|e)rie\s*:?\s*(.+)",
        "CTS_QUALIFICATION": r"Qualification professionnelle\s*:?\s*(.+)",
        "CTS_NUMERO_PERMIS_TRAVAIL": r"permis de travail N(?:°|o)\s*([0-9\-/]+)",
        "CTS_DATE_DELIVRANCE_PERMIS": r"D(?:é|e)livr(?:é|e) le\s*:?\s*(\d{2}/\d{2}/\d{4})",
        "CTS_DATE_DEBUT_VALIDITE_PERMIS": r"valable du\s*(\d{2}/\d{2}/\d{4})",
        "CTS_DATE_FIN_VALIDITE_PERMIS": r"valable du\s*\d{2}/\d{2}/\d{4}\s*au\s*(\d{2}/\d{2}/\d{4})",
        "CTS_PART_TRANSFERABLE": r"part transf(?:é|e)rable\s*:?\s*([0-9][0-9\s.,]*)",
        "CTS_PART_PAYABLE_DZD": r"part payable en dinars alg(?:é|e)rien\s*:?\s*([0-9][0-9\s.,]*)",
        "CTS_NUMERO_SS_PAYS_ORIGINE": r"pays d'origine\s*:?\s*(\d+)",
        "CTS_NUMERO_SS_ALGERIE": r"En Alg(?:é|e)rie\s*:?\s*(\d+)",
        "CTS_DATE_DOCUMENT": r"Fait à\s*:?\s*.+?le\s*(\d{2}/\d{2}/\d{4})",
    },
}

def extract_native(text, doc_type):
    """Extraction regex sur la couche texte d'un PDF né numérique."""
    text = _clean_lines(text)
    patterns = NATIVE_PATTERNS.get(doc_type, {})
    out = {}
    for field, pattern in patterns.items():
        out[field] = _rx(pattern, text) if pattern else None
    if doc_type == "CONTRAT_TRAVAIL":
        out["CTR_TYPE"] = "CONTRAT DE TRAVAIL A DUREE DETERMINEE"
    if doc_type == "CONTRAT_SPECIFIQUE":
        ligne = _rx(r"Salaire mensuel de base net\s*:?\s*(.+)", text)
        out["CTS_LIGNE_SALAIRE_BRUTE"] = ligne
        split = split_ligne_salaire(ligne)
        out["CTS_SALAIRE_NET"] = str(split["nouveau"]) if split["nouveau"] is not None else None
        out["CTS_SALAIRE_NET_ANCIEN"] = str(split["ancien"]) if split["ancien"] is not None else None
        out["CTS_MENTION_AU_LIEU_DE_PRESENTE"] = split["mention_presente"]
    if doc_type == "ENGAGEMENT_DOMICILIATION":
        # DOM_DATE_SIGNATURE est manuscrite -> reste None sur natif aussi.
        out["DOM_DATE_SIGNATURE"] = None
    return out

print("✅ Classification mots-clés + extraction native (regex) OK")


In [ ]:
# =====================================================================
# 5. Normalisation, référentiel externe, ligne de salaire
# =====================================================================
NULL_VALUES = {"", "NULL", "NONE", "N/A", "NA", "NEANT", "NÉANT", "ILLISIBLE"}

def clean_raw_value(value):
    if value is None or isinstance(value, bool):
        return value
    text = str(value).strip()
    return None if text.upper() in NULL_VALUES else text

def clean_raw_dict(data):
    return {k: clean_raw_value(v) for k, v in data.items()} if isinstance(data, dict) else {}

def normalize_amount(value):
    if value is None:
        return None
    text = re.sub(r"[^0-9,.\-]", "", str(value).replace("\xa0", " ").strip())
    if not text:
        return None
    if "," in text and "." in text:
        text = text.replace(".", "").replace(",", ".") if text.rfind(",") > text.rfind(".") else text.replace(",", "")
    elif "," in text:
        text = text.replace(",", ".")
    try:
        return round(float(text), 2)
    except Exception:
        return None

def parse_date(value):
    if value is None:
        return None
    if isinstance(value, pd.Timestamp):
        return value.date()
    if isinstance(value, datetime):
        return value.date()
    if isinstance(value, date):
        return value
    parsed = pd.to_datetime(str(value).strip(), dayfirst=True, errors="coerce")
    return None if pd.isna(parsed) else parsed.date()

def normalize_text(value):
    if value is None:
        return ""
    text = re.sub(r"[^\w\s]", " ", str(value).upper())
    return re.sub(r"\s+", " ", text).strip()

def name_similarity(a, b):
    from difflib import SequenceMatcher
    a, b = normalize_text(a), normalize_text(b)
    return SequenceMatcher(None, a, b).ratio() if a and b else 0.0

def normalize_dom_reference(value, date_domiciliation=None):
    if value is None:
        return None
    raw = str(value).strip().upper()
    compact = re.sub(r"[^A-Z0-9]", "", raw)
    if re.fullmatch(r"271901\d{4}[1-4]40\d{5}[A-Z]{3}", compact):
        return compact
    parts = [p.strip().upper() for p in re.split(r"[|;/\\]+", raw) if p.strip()]
    if len(parts) >= 5:
        prefix = re.sub(r"\D", "", parts[0]); fixed_code = re.sub(r"\D", "", parts[2])
        sequence = re.sub(r"\D", "", parts[3]).zfill(5)
        currency = re.sub(r"[^A-Z]", "", parts[4]) or "DZD"
        match_yq = re.search(r"(\d{4})\D*([1-4])", parts[1])
        if prefix == "271901" and fixed_code == "40" and match_yq:
            return f"271901{match_yq.group(1)}{match_yq.group(2)}40{sequence[:5]}{currency[:3]}"
    match_short = re.search(r"(\d{4})\D*([1-4])\D*40\D*(\d{1,5})(?:\D*([A-Z]{3}))?", raw)
    if match_short:
        return f"271901{match_short.group(1)}{match_short.group(2)}40{match_short.group(3).zfill(5)}{match_short.group(4) or 'DZD'}"
    return compact or None

def dom_reference_is_valid(value):
    return bool(value and re.fullmatch(r"271901\d{4}[1-4]40\d{5}[A-Z]{3}", str(value)))

def first_not_null(*values):
    return next((v for v in values if v not in (None, "", "NULL")), None)

# --- Ligne de salaire "au lieu de" -----------------------------------
MOTIF_MONTANT = r"\d[\d\s\u00a0.,]*\d|\d"
MOTIF_AU_LIEU_DE = r"\ben\s+lieu\s+de\b|\bau\s+lieu\s+d[eu]\b"

def split_ligne_salaire(ligne):
    vide = {"nouveau": None, "ancien": None, "mention_presente": False}
    if not ligne:
        return vide
    texte = str(ligne).replace("\xa0", " ").strip()
    sep = re.search(MOTIF_AU_LIEU_DE, texte, flags=re.I)
    if not sep:
        montants = re.findall(MOTIF_MONTANT, texte)
        return {"nouveau": normalize_amount(montants[0]) if montants else None,
                "ancien": None, "mention_presente": False}
    avant, apres = texte[:sep.start()], texte[sep.end():]
    ma = re.findall(MOTIF_MONTANT, avant)
    mp = re.findall(MOTIF_MONTANT, apres)
    return {"nouveau": normalize_amount(ma[-1]) if ma else None,
            "ancien": normalize_amount(mp[0]) if mp else None,
            "mention_presente": True}

# --- Référentiel DOM / PREDOM -----------------------------------------
ALIASES = {
    "numero_dom": ["Numéro de domiciliation", "Numero de domiciliation", "N° domiciliation"],
    "date_dom": ["Date domiciliation", "Date de domiciliation", "Date demande", "Request Decision Date"],
    "nom_client": ["Nom complet/Raison social", "Nom complet/Raison sociale", "Nom du Fournisseur/Client", "Name", "Nom"],
    "numero_client": ["Identifiant client", "Numero client", "N° client", "Code client"],
    "date_debut": ["Date début du contrat", "Date debut du contrat"],
    "date_fin": ["Date fin de contrat", "Date de fin du contrat"],
    "reference": ["Référence", "Reference"],
}

def safe_read_excel(path):
    path = Path(path)
    if not path.exists():
        return None
    try:
        frames = []
        for sheet in pd.ExcelFile(path).sheet_names:
            df = pd.read_excel(path, sheet_name=sheet)
            if not df.empty:
                df["_SOURCE_SHEET"] = sheet
                frames.append(df)
        return pd.concat(frames, ignore_index=True) if frames else None
    except Exception as exc:
        print(f"⚠️ Lecture impossible {path.name}: {exc}")
        return None

def resolve_column(df, aliases):
    if df is None:
        return None
    normalized = {normalize_text(c): c for c in df.columns}
    for alias in aliases:
        if normalize_text(alias) in normalized:
            return normalized[normalize_text(alias)]
    for alias in aliases:
        target = normalize_text(alias)
        for key, original in normalized.items():
            if target in key or key in target:
                return original
    return None

def prepare_reference(df, source):
    if df is None or df.empty:
        return None
    out = pd.DataFrame()
    out["SOURCE_MATCH"] = source
    out["SOURCE_SHEET"] = df.get("_SOURCE_SHEET")
    for key, aliases in ALIASES.items():
        col = resolve_column(df, aliases)
        out[key] = df[col] if col else None
    out["numero_dom_normalise"] = out.apply(lambda r: normalize_dom_reference(r.get("numero_dom"), r.get("date_dom")), axis=1)
    out["date_debut_parse"] = out["date_debut"].apply(parse_date)
    out["date_fin_parse"] = out["date_fin"].apply(parse_date)
    return out

def _first_non_empty(series):
    values = [v for v in series.tolist()
              if v is not None and not (isinstance(v, float) and pd.isna(v)) and str(v).strip() != ""]
    return values[0] if values else None

def build_reference_table():
    dom = prepare_reference(safe_read_excel(DOM_REFERENCE_EXCEL), "DOM")
    predom = prepare_reference(safe_read_excel(PREDOM_REFERENCE_EXCEL), "PREDOM")
    if dom is None or dom.empty:
        return None
    dom = dom.copy()
    dom["DOM_MATCH_COUNT"] = dom.groupby("numero_dom_normalise")["numero_dom_normalise"].transform("size")
    if predom is None or predom.empty:
        dom["date_debut_predom"] = None; dom["date_fin_predom"] = None
        dom["reference_predom"] = None; dom["PREDOM_MATCH_COUNT"] = 0
        dom["date_debut_match"] = dom["date_debut_parse"]
        dom["date_fin_match"] = dom["date_fin_parse"]
        return dom
    predom_grouped = (
        predom.groupby("numero_dom_normalise", dropna=False)
        .agg(date_debut_predom=("date_debut", _first_non_empty),
             date_fin_predom=("date_fin", _first_non_empty),
             date_debut_predom_parse=("date_debut_parse", _first_non_empty),
             date_fin_predom_parse=("date_fin_parse", _first_non_empty),
             reference_predom=("reference", _first_non_empty),
             PREDOM_MATCH_COUNT=("numero_dom_normalise", "size"))
        .reset_index()
    )
    reference = dom.merge(predom_grouped, on="numero_dom_normalise", how="left", validate="many_to_one")
    reference["PREDOM_MATCH_COUNT"] = reference["PREDOM_MATCH_COUNT"].fillna(0).astype(int)
    reference["date_debut_match"] = reference["date_debut_predom_parse"].where(
        reference["date_debut_predom_parse"].notna(), reference["date_debut_parse"])
    reference["date_fin_match"] = reference["date_fin_predom_parse"].where(
        reference["date_fin_predom_parse"].notna(), reference["date_fin_parse"])
    return reference

print("✅ Normalisation + référentiel OK")


In [ ]:
# =====================================================================
# 6. Prompts V8
# =====================================================================
# Classification : version courte (~100 tokens). La tâche est simple,
# le prefix caching vLLM rend son coût quasi nul de toute façon.
PROMPT_CLASSIFICATION = """Classe cette page en UNE catégorie parmi :
ENGAGEMENT_DOMICILIATION, CONTRAT_TRAVAIL, CONTRAT_SPECIFIQUE,
TITRE_TRAVAIL, PERMIS_TRAVAIL_COUVERTURE, AUTRE.

Indices :
- titre "ENGAGEMENT DE DOMICILIATION" ou "CONTRAT DES SALARIES ETRANGERS"
  -> ENGAGEMENT_DOMICILIATION
- titre contenant "MAIN D'OEUVRE ETRANGERE" -> CONTRAT_SPECIFIQUE
- titre "CONTRAT DE TRAVAIL A DUREE DETERMINEE" -> CONTRAT_TRAVAIL
- bloc identité (photo ou champs Nom / Prénom / Date de naissance) :
  TOUJOURS TITRE_TRAVAIL, même si "Permis de Travail" apparaît en bas
- couverture "Permis de Travail" seule, sans bloc identité
  -> PERMIS_TRAVAIL_COUVERTURE

Réponds uniquement ce JSON :
{"type_document":"CATEGORIE","bloc_identite_present":true}"""

COMMON_RAW_RULES = """Tu analyses une seule image (page entière ou recadrage).
Règles :
1. Extraire uniquement les champs demandés, valeur juste après le libellé
   (même ligne, ou ligne suivante si la valeur continue).
2. Recopier la valeur exactement : espaces, ponctuation, format date/montant.
3. Pas de correction orthographique, pas de normalisation, pas d'invention.
4. Si le libellé est absent ou la valeur illisible : null.
5. Un champ hors de l'image vaut null : ne devine jamais.
6. Retourne uniquement un objet JSON valide, sans commentaire."""

PROMPT_ENGAGEMENT = COMMON_RAW_RULES + """
TYPE : ENGAGEMENT_DOMICILIATION
{
  "DOM_NOM_RAISON_SOCIAL_CLIENT": null,
  "DOM_COMPTE_LOCAL": null,
  "DOM_ADRESSE_CLIENT": null,
  "DOM_AGENCE_DOMICILIATAIRE": null,
  "DOM_NUMERO_CONTRAT": null,
  "DOM_DUREE_CONTRAT_MOIS": null,
  "DOM_DATE_DEBUT_CONTRAT": null,
  "DOM_DATE_FIN_CONTRAT": null,
  "DOM_NOM_RAISON_SOCIAL_EMPLOYEUR": null,
  "DOM_ADRESSE_EMPLOYEUR": null,
  "DOM_SALAIRE_NET_MENSUEL": null,
  "DOM_PART_TRANSFERABLE": null,
  "DOM_TAUX_TRANSFERABLE": null,
  "DOM_MONTANT_TOTAL_DOMICILIE": null,
  "DOM_DATE_SIGNATURE": null
}
Libellés :
- DOM_NOM_RAISON_SOCIAL_CLIENT : après "Nom et raison social" (deux colonnes
  possibles : recopier nom + prénom séparés par un espace).
- DOM_COMPTE_LOCAL : après "N de compte" / "N° de compte".
- DOM_ADRESSE_CLIENT : première "Adresse" de la section Identification du client.
- DOM_AGENCE_DOMICILIATAIRE : après "Agence domiciliataire" (haut de page).
- DOM_NUMERO_CONTRAT / DOM_DUREE_CONTRAT_MOIS : "Numéro du contrat" /
  "Durée du contrat" (ou "Duré").
- DOM_DATE_DEBUT_CONTRAT / DOM_DATE_FIN_CONTRAT : dates correspondantes.
- DOM_NOM_RAISON_SOCIAL_EMPLOYEUR : après "Nom et raison sociale de L'Employeur".
- DOM_ADRESSE_EMPLOYEUR : après "Adresse de L'Employeur" (ligne suivante si
  l'adresse continue).
- DOM_SALAIRE_NET_MENSUEL : après "Salaire net mensuel".
- DOM_PART_TRANSFERABLE : après "Montant de la part transférable".
- DOM_TAUX_TRANSFERABLE : après "Pourcentage en regard du salaire net mensuel".
- DOM_MONTANT_TOTAL_DOMICILIE : après "Montant domicilié en DZD" (souvent vide).
- DOM_DATE_SIGNATURE : date près de "lu et approuvé" en bas de page."""

PROMPT_CONTRAT = COMMON_RAW_RULES + """
TYPE : CONTRAT_TRAVAIL
{
  "CTR_REFERENCE_DOCUMENT": null,
  "CTR_TYPE": null,
  "CTR_EMPLOYEUR": null,
  "CTR_ACTIVITE_EMPLOYEUR": null,
  "CTR_DUREE_MOIS": null,
  "CTR_DATE_DEBUT_CONTRAT": null,
  "CTR_POSTE": null,
  "CTR_NOM_PRENOM_TRAVAILLEUR": null,
  "CTR_PERE_NOM_PRENOM": null,
  "CTR_MERE_NOM_PRENOM": null,
  "CTR_NATIONALITE": null,
  "CTR_DATE_NAISSANCE": null,
  "CTR_LIEU_PAYS_NAISSANCE": null,
  "CTR_ADRESSE_ALGERIE": null,
  "CTR_QUALIFICATION": null,
  "CTR_NUMERO_PERMIS_TRAVAIL": null,
  "CTR_DATE_DELIVRANCE_PERMIS": null,
  "CTR_DATE_DEBUT_VALIDITE_PERMIS": null,
  "CTR_DATE_FIN_VALIDITE_PERMIS": null,
  "CTR_SALAIRE_BRUT": null,
  "CTR_SALAIRE_NET": null,
  "CTR_AFFILIATION_SS": null,
  "CTR_NUMERO_EMPLOYEUR": null,
  "CTR_DATE_SIGNATURE": null
}
Libellés :
- CTR_REFERENCE_DOCUMENT : référence en haut à gauche, ex. "TR-6166".
- CTR_TYPE : titre complet du document.
- CTR_EMPLOYEUR : après "au nom de l'employeur ci-après désigné :".
- CTR_ACTIVITE_EMPLOYEUR : après "Nature de l'activité :".
- CTR_DUREE_MOIS : après "pour une durée de :" (avant "Mois").
- CTR_DATE_DEBUT_CONTRAT : après "à compter du :".
- CTR_POSTE : après "En qualité de :".
- CTR_NOM_PRENOM_TRAVAILLEUR : après "A (Mr/Mme) :".
- CTR_PERE_NOM_PRENOM : après "Fils de :" avant "et de :".
- CTR_MERE_NOM_PRENOM : après "et de :".
- CTR_NATIONALITE : après "Nationalité :".
- CTR_DATE_NAISSANCE : après "Né(e) le :" avant "à".
- CTR_LIEU_PAYS_NAISSANCE : après "à" sur la ligne de naissance.
- CTR_ADRESSE_ALGERIE : après "Adresse en Algérie :".
- CTR_QUALIFICATION : après "Qualification professionnelle :".
- CTR_NUMERO_PERMIS_TRAVAIL : après "permis de travail N°" (référence complète
  avec la partie après "/").
- CTR_DATE_DELIVRANCE_PERMIS : après "Délivré le :".
- CTR_DATE_DEBUT_VALIDITE_PERMIS / CTR_DATE_FIN_VALIDITE_PERMIS : première date
  après "Valable du" / date après "au".
- CTR_SALAIRE_BRUT / CTR_SALAIRE_NET : après "Montant du salaire mensuel brut :"
  / "Montant du salaire mensuel net :".
- CTR_AFFILIATION_SS : après "Affiliation à la sécurité sociale :".
- CTR_NUMERO_EMPLOYEUR : après "Employeur :".
- CTR_DATE_SIGNATURE : date après "Fait à : Bethioua, le"."""

PROMPT_CONTRAT_SPECIFIQUE = COMMON_RAW_RULES + """
TYPE : CONTRAT_SPECIFIQUE
{
  "CTS_REFERENCE_DOCUMENT": null,
  "CTS_SAP_ID": null,
  "CTS_EMPLOYEUR": null,
  "CTS_ACTIVITE_EMPLOYEUR": null,
  "CTS_DUREE_MOIS": null,
  "CTS_DATE_DEBUT_CONTRAT": null,
  "CTS_POSTE": null,
  "CTS_NOM_PRENOM_TRAVAILLEUR": null,
  "CTS_PERE_NOM_PRENOM": null,
  "CTS_MERE_NOM_PRENOM": null,
  "CTS_NATIONALITE": null,
  "CTS_DATE_NAISSANCE": null,
  "CTS_LIEU_PAYS_NAISSANCE": null,
  "CTS_ADRESSE_ALGERIE": null,
  "CTS_QUALIFICATION": null,
  "CTS_NUMERO_PERMIS_TRAVAIL": null,
  "CTS_DATE_DELIVRANCE_PERMIS": null,
  "CTS_DATE_DEBUT_VALIDITE_PERMIS": null,
  "CTS_DATE_FIN_VALIDITE_PERMIS": null,
  "CTS_LIGNE_SALAIRE_BRUTE": null,
  "CTS_SALAIRE_NET": null,
  "CTS_SALAIRE_NET_ANCIEN": null,
  "CTS_MENTION_AU_LIEU_DE_PRESENTE": null,
  "CTS_PART_TRANSFERABLE": null,
  "CTS_PART_PAYABLE_DZD": null,
  "CTS_NUMERO_SS_PAYS_ORIGINE": null,
  "CTS_NUMERO_SS_ALGERIE": null,
  "CTS_DATE_DOCUMENT": null
}
Mêmes libellés que le contrat de travail, avec :
- CTS_SAP_ID : après "SAP id -" en haut de page.
- CTS_POSTE : après "en qualité de :".
- LIGNE DE SALAIRE (règle particulière), deux formes possibles :
  A : "Salaire mensuel de base net : 506,471.38"
  B : "Salaire mensuel de base net : 506,471.38 au lieu de 479,274.29"
  - CTS_LIGNE_SALAIRE_BRUTE : ligne ENTIÈRE recopiée telle quelle.
  - CTS_SALAIRE_NET : PREMIER montant (celui avant "au lieu de") = nouveau.
  - CTS_SALAIRE_NET_ANCIEN : SECOND montant (après "au lieu de") ; null sinon.
  - CTS_MENTION_AU_LIEU_DE_PRESENTE : true si "au lieu de" présent.
- CTS_PART_TRANSFERABLE : après "La part transférable :".
- CTS_PART_PAYABLE_DZD : après "La part payable en dinars algérien :".
- CTS_NUMERO_SS_PAYS_ORIGINE : après "Dans le pays d'origine :".
- CTS_NUMERO_SS_ALGERIE : après "En Algérie :".
- CTS_DATE_DOCUMENT : date après "Fait à : Bethioua, le"."""

PROMPT_TITRE_TRAVAIL = COMMON_RAW_RULES + """
TYPE : TITRE_TRAVAIL — titre algérien bilingue arabe/français, scan dégradé.
DEUX COLONNES :
  GAUCHE (poste/employeur), schéma : libellé français ..... valeur ..... arabe
  DROITE (identité, à côté de la photo), même schéma.
Ne confonds JAMAIS un libellé arabe avec une valeur : la valeur est le texte
latin sur les pointillés, entre le libellé français et le libellé arabe.
{
  "TTR_NUMERO_PERMIS": null,
  "TTR_NUMERO_MANUSCRIT": null,
  "TTR_POSTE": null,
  "TTR_DUREE": null,
  "TTR_DATE_DEBUT": null,
  "TTR_DATE_FIN": null,
  "TTR_LIEU_TRAVAIL": null,
  "TTR_EMPLOYEUR": null,
  "TTR_ADRESSE_EMPLOYEUR": null,
  "TTR_FAIT_A": null,
  "TTR_DATE_DELIVRANCE": null,
  "TTR_NOM": null,
  "TTR_PRENOM": null,
  "TTR_DATE_NAISSANCE": null,
  "TTR_LIEU_NAISSANCE": null,
  "TTR_PAYS": null,
  "TTR_NATIONALITE": null,
  "TTR_QUALIFICATION": null,
  "TTR_DATE_ENTREE_ALGERIE": null
}
Libellés :
- TTR_NUMERO_PERMIS : référence encadrée en haut à gauche
  "21-00002974 / 31-25-001448" (complète, sans parenthèses).
- TTR_NUMERO_MANUSCRIT : nombre manuscrit sous la référence ; null si absent.
- TTR_POSTE : sous "autorisé à occuper le poste de travail de" (2-3 lignes
  possibles : recopier l'ensemble séparé par des espaces).
- TTR_DUREE : après "Durée" ("2 ANS, 0 JOURS").
- TTR_DATE_DEBUT / TTR_DATE_FIN : après "Du" / après "Au" (attention : "Au",
  pas "Fin de travail").
- TTR_LIEU_TRAVAIL : après "Lieu de travail" (peut tenir sur 2 lignes).
- TTR_EMPLOYEUR : après "Nom de l'organisme employeur" (2 lignes possibles).
- TTR_ADRESSE_EMPLOYEUR : après "Adresse de l'organisme employeur".
- TTR_FAIT_A : après "Fait à". TTR_DATE_DELIVRANCE : après "Le" sous "Fait à"
  (souvent sous cachet : null si illisible, ne devine pas).
- TTR_NOM / TTR_PRENOM : après "Nom" / "Prénom" (colonne droite).
- TTR_DATE_NAISSANCE / TTR_LIEU_NAISSANCE : après les libellés correspondants.
- TTR_PAYS / TTR_NATIONALITE / TTR_QUALIFICATION : idem (qualification pouvant
  couper sur 2 lignes).
- TTR_DATE_ENTREE_ALGERIE : après "Date d'entrée en Algérie"."""

PROMPT_PERMIS_COUVERTURE = COMMON_RAW_RULES + """
TYPE : PERMIS_TRAVAIL_COUVERTURE (couverture "Permis de Travail").
{
  "PTR_NUMERO_SERIE": null,
  "PTR_WILAYA": null
}
- PTR_NUMERO_SERIE : numéro de série en bas (après "N° de Série") ; null si
  illisible.
- PTR_WILAYA : après "Direction de l'Emploi de la Wilaya de :".
Ne pas extraire le texte des extraits de loi."""

PROMPTS_EXTRACTION = {
    "ENGAGEMENT_DOMICILIATION": PROMPT_ENGAGEMENT,
    "CONTRAT_TRAVAIL": PROMPT_CONTRAT,
    "CONTRAT_SPECIFIQUE": PROMPT_CONTRAT_SPECIFIQUE,
    "TITRE_TRAVAIL": PROMPT_TITRE_TRAVAIL,
    "PERMIS_TRAVAIL_COUVERTURE": PROMPT_PERMIS_COUVERTURE,
}
TYPES_VALIDES = set(PROMPTS_EXTRACTION) | {"AUTRE"}
TYPES_DACTYLO = {"ENGAGEMENT_DOMICILIATION", "CONTRAT_TRAVAIL", "CONTRAT_SPECIFIQUE"}
TYPES_PERMIS = {"TITRE_TRAVAIL", "PERMIS_TRAVAIL_COUVERTURE"}

def champs_attendus_depuis_prompt(prompt):
    m = re.search(r"\{[^{}]*\}", prompt, flags=re.S)
    if not m:
        return []
    try:
        return list(json.loads(m.group(0)).keys())
    except Exception:
        return re.findall(r'"([A-Z0-9_]+)"\s*:', m.group(0))

CHAMPS_ATTENDUS = {t: champs_attendus_depuis_prompt(p) for t, p in PROMPTS_EXTRACTION.items()}

# Stratégies d'extraction (recadrages en fractions de page)
STRATEGIES_PERMIS = [
    {"nom": "HD_BLOC_TITRE", "cle": "titre"},
    {"nom": "HD_COLONNE_IDENTITE", "cle": "colonne_identite"},
    {"nom": "HD_COLONNE_POSTE", "cle": "colonne_poste"},
    {"nom": "PAGE_ENTIERE_HD", "crop": (0.00, 1.00, 0.00, 1.00)},
]

print("✅ Prompts V8 chargés |", {t: len(c) for t, c in CHAMPS_ATTENDUS.items()})


In [ ]:
# =====================================================================
# 7. Moteurs vLLM (cascade : léger -> lourd)
# =====================================================================
# Chargement SEQUENTIEL : jamais deux LLM en même temps en mémoire
# (évite les conflits NCCL et la pression VRAM). Passe 1 = 7B, puis
# libération, puis 35B-A3B-FP8 pour planche permis + escalades.

llm_light = None
llm_heavy = None

def load_engine_light():
    global llm_light
    print("Chargement moteur LÉGER (Qwen2.5-VL-7B)..."); t0 = time.time()
    llm_light = LLM(
        model=MODELE_LEGER_PATH,
        dtype="bfloat16",
        gpu_memory_utilization=0.55,     # laisse de la marge pour les images
        max_model_len=6144,              # image ~1500 tok + prompt + sortie 500
        max_num_seqs=MAX_NUM_SEQS,
        enable_prefix_caching=True,      # les prompts d'extraction sont identiques
        trust_remote_code=True,
        limit_mm_per_prompt={"image": 1},
        disable_log_stats=True,
    )
    print(f"✅ Moteur léger chargé en {time.time()-t0:.0f}s")

def load_engine_heavy():
    global llm_heavy
    print("Chargement moteur LOURD (Qwen3.6-35B-A3B-FP8)..."); t0 = time.time()
    llm_heavy = LLM(
        model=MODELE_LOURD_PATH,
        quantization="fp8",              # poids FP8 natifs, pas de déquant bf16
        dtype="auto",
        gpu_memory_utilization=0.85,
        max_model_len=12288,             # planche permis HD ~5000 tok image
        max_num_seqs=16,                 # MoE : séquences plus lourdes
        enable_prefix_caching=True,
        trust_remote_code=True,
        limit_mm_per_prompt={"image": 1},
        disable_log_stats=True,
    )
    print(f"✅ Moteur lourd chargé en {time.time()-t0:.0f}s")

def unload_engine(which):
    global llm_light, llm_heavy
    eng = llm_light if which == "light" else llm_heavy
    if eng is None:
        return
    del eng
    if which == "light":
        llm_light = None
    else:
        llm_heavy = None
    gc.collect(); torch.cuda.empty_cache()
    print(f"🧹 Moteur {which} libéré | VRAM libre : {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

print("✅ Gestion des moteurs OK")


In [ ]:
# =====================================================================
# 8. Helpers d'inférence vLLM (batch, data-URI, désactivation thinking)
# =====================================================================
def pil_to_data_uri(img):
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()

def vlm_chat_batch(llm, prompt, images, max_tokens, max_pixels):
    """Un appel llm.chat batché pour N images identiques-prompt.
    Retourne (textes, tokens_sortie, tokens_entree_estimes)."""
    conversations = [
        [{"role": "user", "content": [
            {"type": "image_url", "image_url": {"url": pil_to_data_uri(img)}},
            {"type": "text", "text": prompt},
        ]}]
        for img in images
    ]
    sp = SamplingParams(temperature=0.0, max_tokens=max_tokens, seed=0)
    kwargs = {"mm_processor_kwargs": {"min_pixels": 256 * 28 * 28, "max_pixels": max_pixels}}
    try:
        outs = llm.chat(conversations, sp, **kwargs)
    except TypeError:
        outs = llm.chat(conversations, sp)   # vLLM ancien sans mm_processor_kwargs
    texts = [o.outputs[0].text for o in outs]
    tokens_out = [len(o.outputs[0].token_ids) for o in outs]
    prompt_tokens_est = int(len(prompt) / 3.5)
    tokens_in = [estimate_image_tokens(img) + prompt_tokens_est for img in images]
    return texts, tokens_out, tokens_in

def vlm_classify_pages(records_scanned):
    """Classification VLM (moteur léger) d'un lot de pages scannées."""
    images = [r["image"] for r in records_scanned]
    texts, tokens_out, tokens_in = vlm_chat_batch(
        llm_light, PROMPT_CLASSIFICATION, images,
        MAX_NEW_TOKENS_CLASSIFICATION, 1280 * 28 * 28)
    for rec, text, tout, tin in zip(records_scanned, texts, tokens_out, tokens_in):
        parsed = parse_json_response(text)
        doc_type = parsed.get("type_document") or "AUTRE"
        bloc_identite = bool(parsed.get("bloc_identite_present"))
        if doc_type not in TYPES_VALIDES:
            doc_type = "AUTRE"
        # Correctif planche : bloc identité -> TITRE_TRAVAIL, toujours
        if bloc_identite and doc_type in ("PERMIS_TRAVAIL_COUVERTURE", "AUTRE"):
            doc_type = "TITRE_TRAVAIL"
        rec.update({
            "doc_type": doc_type,
            "bloc_identite_present": bloc_identite,
            "classification_method": "VLM_7B",
            "classification_confidence": 1.0 if doc_type != "AUTRE" else 0.5,
            "classification_tokens_in": tin,
            "classification_tokens_out": tout,
        })
    return records_scanned

def vlm_extract_grouped(records, doc_type):
    """Extraction VLM (moteur léger) d'un groupe de pages de même type.
    Fusion des résultats dans record['raw_data']. Retourne les records
    dont le taux de remplissage reste sous le seuil (escalade)."""
    if not records:
        return []
    prompt = PROMPTS_EXTRACTION[doc_type]
    images = [r["image"] for r in records]
    texts, tokens_out, tokens_in = vlm_chat_batch(
        llm_light, prompt, images,
        MAX_NEW_TOKENS_EXTRACTION_TYPED, 1400 * 28 * 28)
    escalades = []
    champs = CHAMPS_ATTENDUS.get(doc_type) or []
    for rec, text, tout, tin in zip(records, texts, tokens_out, tokens_in):
        parsed = clean_raw_dict(parse_json_response(text))
        for c in champs:
            parsed.setdefault(c, None)
        rec["raw_data"] = parsed
        rec["extraction_method"] = "VLM_7B"
        rec["extraction_taux_remplissage"] = taux_remplissage(parsed, champs)
        rec["extraction_tokens_in"] = tin
        rec["extraction_tokens_out"] = tout
        rec["extraction_status"] = ("OK" if rec["extraction_taux_remplissage"] >= SEUIL_REMPLISSAGE_OK
                                    else "PARTIELLE")
        if rec["extraction_taux_remplissage"] < SEUIL_REMPLISSAGE_OK:
            escalades.append(rec)
    return escalades

print("✅ Inférence batchée (classification + extraction groupee) OK")


In [ ]:
# =====================================================================
# 9. Passe lourde : planche permis (HD recadrée) + escalades
# =====================================================================
def heavy_extract_permis(items):
    """items : dicts {pdf_path, page_index, record, frontiere, doc_type}.
    Stratégies appliquées par vagues batchées : on s'arrête pour une page
    dès que le seuil est atteint ; une passe ne complète que les champs vides."""
    pending = list(items)
    for strategie in STRATEGIES_PERMIS:
        if not pending:
            break
        todo = [it for it in pending
                if taux_remplissage(it["record"]["raw_data"], CHAMPS_ATTENDUS[it["doc_type"]]) < SEUIL_REMPLISSAGE_OK]
        if not todo:
            break
        images, group = [], []
        for it in todo:
            crop = it["crops"][strategie["cle"]] if "cle" in strategie else strategie.get("crop")
            img = render_page_region(it["pdf_path"], it["page_index"], crop=crop)
            images.append(img)
            group.append((it, img))
        prompt = PROMPTS_EXTRACTION[group[0][0]["doc_type"]]
        texts, tokens_out, tokens_in = vlm_chat_batch(
            llm_heavy, prompt, images,
            MAX_NEW_TOKENS_EXTRACTION_PERMIS, 2600 * 28 * 28)
        for (it, img), text, tout, tin in zip(group, texts, tokens_out, tokens_in):
            parsed = clean_raw_dict(parse_json_response(text))
            fusion = it["record"]["raw_data"]
            ajouts = 0
            for cle, valeur in parsed.items():
                if fusion.get(cle) in (None, "") and valeur not in (None, ""):
                    fusion[cle] = valeur
                    ajouts += 1
            rec = it["record"]
            rec["extraction_method"] = "VLM_35B_A3B"
            rec["extraction_strategies"] = (rec.get("extraction_strategies") or []) + [strategie["nom"]]
            rec["extraction_tokens_in"] = rec.get("extraction_tokens_in", 0) + tin
            rec["extraction_tokens_out"] = rec.get("extraction_tokens_out", 0) + tout
            champs = CHAMPS_ATTENDUS[rec["doc_type"]]
            for c in champs:
                fusion.setdefault(c, None)
            rec["raw_data"] = fusion
            rec["extraction_taux_remplissage"] = taux_remplissage(fusion, champs)
            rec["extraction_status"] = ("OK" if rec["extraction_taux_remplissage"] >= SEUIL_REMPLISSAGE_OK
                                        else "PARTIELLE")
    for it in items:
        rec = it["record"]
        champs = CHAMPS_ATTENDUS[rec["doc_type"]]
        for c in champs:
            rec["raw_data"].setdefault(c, None)
        rec["extraction_taux_remplissage"] = taux_remplissage(rec["raw_data"], champs)
        if rec["extraction_status"] not in ("OK", "PARTIELLE"):
            rec["extraction_status"] = ("OK" if rec["extraction_taux_remplissage"] >= SEUIL_REMPLISSAGE_OK
                                        else "PARTIELLE")

def heavy_extract_escalades(records):
    """Reprise des pages dactylo sous-remplies par le 7B, page entière HD."""
    if not records:
        return
    doc_types = sorted({r["doc_type"] for r in records})
    for doc_type in doc_types:
        group = [r for r in records if r["doc_type"] == doc_type]
        images = []
        for r in group:
            images.append(render_page_region(r["_pdf_path"], r["page_num"] - 1,
                                             zoom=3.2, max_side=1600))
        prompt = PROMPTS_EXTRACTION[doc_type]
        texts, tokens_out, tokens_in = vlm_chat_batch(
            llm_heavy, prompt, images,
            MAX_NEW_TOKENS_EXTRACTION_TYPED, 1800 * 28 * 28)
        champs = CHAMPS_ATTENDUS[doc_type]
        for rec, text, tout, tin in zip(group, texts, tokens_out, tokens_in):
            parsed = clean_raw_dict(parse_json_response(text))
            fusion = rec["raw_data"]
            for cle, valeur in parsed.items():
                if fusion.get(cle) in (None, "") and valeur not in (None, ""):
                    fusion[cle] = valeur
            for c in champs:
                fusion.setdefault(c, None)
            rec["raw_data"] = fusion
            rec["extraction_method"] = "VLM_7B puis 35B_A3B"
            rec["extraction_status"] = ("OK" if taux_remplissage(fusion, champs) >= SEUIL_REMPLISSAGE_OK
                                        else "PARTIELLE")
            rec["extraction_tokens_in"] += tin
            rec["extraction_tokens_out"] += tout

print("✅ Passe lourde (permis + escalades) OK")


In [ ]:
# =====================================================================
# 10. Consolidation dossier, matching, planning P1/P2
# =====================================================================
AMOUNT_FIELDS = {
    "DOM_SALAIRE_NET_MENSUEL", "DOM_PART_TRANSFERABLE",
    "DOM_MONTANT_TOTAL_DOMICILIE", "CTR_SALAIRE_BRUT",
    "CTR_SALAIRE_NET", "CTS_SALAIRE_NET", "CTS_SALAIRE_NET_ANCIEN",
    "CTS_PART_TRANSFERABLE", "CTS_PART_PAYABLE_DZD",
}

def build_page_row(pdf_name, record):
    row = {
        "FICHIER": pdf_name,
        "PAGE": record.get("page_num"),
        "TYPE_DOCUMENT": record.get("doc_type"),
        "CLASSIFICATION_METHOD": record.get("classification_method"),
        "EXTRACTION_METHOD": record.get("extraction_method"),
        "STATUT_EXTRACTION": record.get("extraction_status"),
        "TAUX_REMPLISSAGE": record.get("extraction_taux_remplissage"),
        "STRATEGIES_UTILISEES": " > ".join(record.get("extraction_strategies") or []) or None,
    }
    row.update(record.get("raw_data") or {})
    return row

def consolidate_dossier(pdf_name, records):
    row = {
        "FICHIER": pdf_name,
        "NB_PAGES": len(records),
        "TYPES_DOCUMENTS": " | ".join(str(r.get("doc_type")) for r in records),
    }
    pages_by_type = defaultdict(list)
    for record in records:
        pages_by_type[record.get("doc_type")].append(str(record.get("page_num")))
        for key, value in (record.get("raw_data") or {}).items():
            if row.get(key) in (None, ""):
                row[key] = value
    for doc_type, pages in pages_by_type.items():
        row[f"PAGES_{doc_type}"] = ",".join(pages)

    for field in AMOUNT_FIELDS:
        if field in row:
            row[field + "_RAW"] = row[field]
            row[field] = normalize_amount(row[field])

    raw_ref = first_not_null(row.get("CTR_REFERENCE_DOMICILIATION"))
    row["REFERENCE_DOM_EXTRAITE_RAW"] = raw_ref
    row["REFERENCE_DOM_EXTRAITE_NORMALISEE"] = normalize_dom_reference(raw_ref)
    row["REFERENCE_DOM_FORMAT_VALIDE"] = dom_reference_is_valid(row["REFERENCE_DOM_EXTRAITE_NORMALISEE"])
    row["NOM_CLIENT_REFERENCE"] = first_not_null(
        row.get("DOM_NOM_RAISON_SOCIAL_CLIENT"),
        row.get("CTR_NOM_PRENOM_TRAVAILLEUR"),
        row.get("CTS_NOM_PRENOM_TRAVAILLEUR"),
        " ".join(x for x in [str(row.get("TTR_NOM") or "").strip(),
                             str(row.get("TTR_PRENOM") or "").strip()] if x) or None)
    row["NUMERO_CONTRAT_REFERENCE"] = first_not_null(row.get("DOM_NUMERO_CONTRAT"),
                                                     row.get("CTR_REFERENCE_DOCUMENT"),
                                                     row.get("CTS_REFERENCE_DOCUMENT"))
    row["DATE_DEBUT_CONTRAT_REFERENCE"] = first_not_null(row.get("DOM_DATE_DEBUT_CONTRAT"),
                                                         row.get("CTR_DATE_DEBUT_CONTRAT"),
                                                         row.get("CTS_DATE_DEBUT_CONTRAT"))
    row["DATE_FIN_CONTRAT_REFERENCE"] = first_not_null(row.get("DOM_DATE_FIN_CONTRAT"),
                                                       row.get("CTR_DATE_FIN_CONTRAT"),
                                                       row.get("CTS_DATE_FIN_CONTRAT"))
    row["NUMERO_PERMIS_REFERENCE"] = first_not_null(
        row.get("TTR_NUMERO_PERMIS"), row.get("CTR_NUMERO_PERMIS_TRAVAIL"),
        row.get("CTS_NUMERO_PERMIS_TRAVAIL"), row.get("PTR_NUMERO_SERIE"))

    # --- Augmentation de salaire ("au lieu de") ----------------------
    salaire_nouveau = first_not_null(row.get("CTS_SALAIRE_NET"),
                                     row.get("CTR_SALAIRE_NET"),
                                     row.get("DOM_SALAIRE_NET_MENSUEL"))
    salaire_ancien = row.get("CTS_SALAIRE_NET_ANCIEN")
    if salaire_ancien is None:
        secours = split_ligne_salaire(row.get("CTS_LIGNE_SALAIRE_BRUTE"))
        if secours["ancien"] is not None:
            salaire_ancien = secours["ancien"]
            row["CTS_SALAIRE_NET_ANCIEN"] = salaire_ancien
            row["SOURCE_AUGMENTATION"] = "RELECTURE_LIGNE_BRUTE"
            if salaire_nouveau is None and secours["nouveau"] is not None:
                salaire_nouveau = secours["nouveau"]
    elif salaire_ancien is not None:
        row["SOURCE_AUGMENTATION"] = "CHAMPS_MODELE"

    row["SALAIRE_NOUVEAU_AUGMENTE"] = salaire_nouveau
    row["SALAIRE_ANCIEN"] = salaire_ancien
    row["AUGMENTATION_DETECTEE"] = bool(salaire_ancien is not None
                                        and salaire_nouveau is not None
                                        and salaire_nouveau != salaire_ancien)
    if row["AUGMENTATION_DETECTEE"]:
        ecart = round(float(salaire_nouveau) - float(salaire_ancien), 2)
        row["MONTANT_AUGMENTATION"] = ecart
        row["SENS_VARIATION_SALAIRE"] = "AUGMENTATION" if ecart > 0 else "DIMINUTION"
        row["TAUX_AUGMENTATION_PCT"] = (round(ecart / float(salaire_ancien) * 100, 2)
                                        if float(salaire_ancien) else None)
    else:
        row["MONTANT_AUGMENTATION"] = None
        row["SENS_VARIATION_SALAIRE"] = None
        row["TAUX_AUGMENTATION_PCT"] = None
        row.setdefault("SOURCE_AUGMENTATION", None)

    salaire_dom = row.get("DOM_SALAIRE_NET_MENSUEL")
    if salaire_dom is not None and salaire_nouveau is not None:
        row["ECART_SALAIRE_DOM_CONTRAT"] = round(float(salaire_dom) - float(salaire_nouveau), 2)
        row["COHERENCE_SALAIRE_DOM_CONTRAT"] = abs(row["ECART_SALAIRE_DOM_CONTRAT"]) < 0.01
    else:
        row["ECART_SALAIRE_DOM_CONTRAT"] = None
        row["COHERENCE_SALAIRE_DOM_CONTRAT"] = None
    row["ALERTE_DOM_SUR_ANCIEN_SALAIRE"] = bool(
        row["AUGMENTATION_DETECTEE"] and salaire_dom is not None and salaire_ancien is not None
        and abs(float(salaire_dom) - float(salaire_ancien)) < 0.01)

    # --- Identité consolidée ------------------------------------------
    row["NOM_TRAVAILLEUR_REFERENCE"] = row["NOM_CLIENT_REFERENCE"]
    row["DATE_NAISSANCE_REFERENCE"] = first_not_null(row.get("CTR_DATE_NAISSANCE"),
                                                     row.get("CTS_DATE_NAISSANCE"),
                                                     row.get("TTR_DATE_NAISSANCE"))
    row["NATIONALITE_REFERENCE"] = first_not_null(row.get("CTR_NATIONALITE"),
                                                  row.get("CTS_NATIONALITE"),
                                                  row.get("TTR_NATIONALITE"))
    row["DATE_ENTREE_ALGERIE"] = row.get("TTR_DATE_ENTREE_ALGERIE")
    row["PERMIS_TRAVAIL_LU"] = bool(row.get("TTR_NOM") or row.get("TTR_NUMERO_PERMIS"))
    return row

def match_dossier(row, ref):
    result = {
        "MATCH_SOURCE": None, "MATCH_METHOD": None, "MATCH_SCORE": 0.00,
        "MATCH_STATUS": "AUCUN_MATCH", "MATCH_CANDIDATES_COUNT": 0,
        "NUMERO_CLIENT_RETENU": None, "NUMERO_DOM_RETENU": None,
        "DATE_DOM_RETENUE": None, "REFERENCE_EXTERNE_RETENUE": None,
        "REFERENCE_PREDOM_RETENUE": None,
        "DATE_DEBUT_CONTRAT_PREDOM": None, "DATE_FIN_CONTRAT_PREDOM": None,
        "PREDOM_TROUVEE": False,
    }
    if ref is None or ref.empty:
        result["MATCH_STATUS"] = "REFERENTIEL_ABSENT"
        return result
    dom_ref = row.get("REFERENCE_DOM_EXTRAITE_NORMALISEE")
    if dom_ref:
        exact = ref[ref["numero_dom_normalise"] == dom_ref].copy()
        result["MATCH_CANDIDATES_COUNT"] = int(len(exact))
        result["MATCH_METHOD"] = "NUMERO_DOM_EXACT"
        if len(exact) == 0:
            result["MATCH_STATUS"] = "NUMERO_DOM_NON_TROUVE"
            return result
        if len(exact) > 1 or int(exact.iloc[0].get("DOM_MATCH_COUNT") or 1) > 1:
            result["MATCH_STATUS"] = "PLUSIEURS_CANDIDATS_DOM"
            return result
        best = exact.iloc[0]
    else:
        start_date = parse_date(row.get("DATE_DEBUT_CONTRAT_REFERENCE"))
        end_date = parse_date(row.get("DATE_FIN_CONTRAT_REFERENCE"))
        result["MATCH_METHOD"] = "PERIODE_CONTRAT_EXACTE"
        if not start_date or not end_date:
            result["MATCH_STATUS"] = "DATES_CONTRAT_INSUFFISANTES"
            return result
        period = ref[(ref["date_debut_match"] == start_date) & (ref["date_fin_match"] == end_date)].copy()
        result["MATCH_CANDIDATES_COUNT"] = int(len(period))
        if len(period) == 0:
            result["MATCH_STATUS"] = "AUCUN_MATCH_PERIODE"
            return result
        if len(period) > 1 or int(period.iloc[0].get("DOM_MATCH_COUNT") or 1) > 1:
            result["MATCH_STATUS"] = "PLUSIEURS_CANDIDATS_PERIODE"
            return result
        best = period.iloc[0]
    predom_count = int(best.get("PREDOM_MATCH_COUNT") or 0)
    result.update({
        "MATCH_SOURCE": "DOM", "MATCH_SCORE": 100.00,
        "MATCH_STATUS": "DOMICILIATION_TROUVEE" if dom_ref else "MATCH_PERIODE_EXACTE",
        "NUMERO_CLIENT_RETENU": best.get("numero_client"),
        "NUMERO_DOM_RETENU": best.get("numero_dom_normalise") or best.get("numero_dom"),
        "DATE_DOM_RETENUE": best.get("date_dom"),
        "REFERENCE_EXTERNE_RETENUE": best.get("reference"),
        "REFERENCE_PREDOM_RETENUE": best.get("reference_predom"),
        "DATE_DEBUT_CONTRAT_PREDOM": best.get("date_debut_predom"),
        "DATE_FIN_CONTRAT_PREDOM": best.get("date_fin_predom"),
        "PREDOM_TROUVEE": predom_count >= 1,
    })
    if predom_count > 1:
        result["MATCH_STATUS"] = "MATCH_DOM_TROUVE_PREDOM_MULTIPLE"
    return result

def month_segment_rows(row):
    start = parse_date(row.get("DATE_DEBUT_CONTRAT_REFERENCE"))
    end = parse_date(row.get("DATE_FIN_CONTRAT_REFERENCE"))
    if not start or not end or end < start:
        return []
    plafond = row.get("DOM_PART_TRANSFERABLE")
    rows, cursor = [], date(start.year, start.month, 1)
    while cursor <= end:
        days_month = calendar.monthrange(cursor.year, cursor.month)[1]
        month_end = date(cursor.year, cursor.month, days_month)
        seg_start, seg_end = max(start, cursor), min(end, month_end)
        if seg_start <= seg_end:
            mois_complet = (seg_start == cursor and seg_end == month_end)
            if mois_complet:
                part = None
                period = f"{cursor.year:04d}-{cursor.month:02d}"
            elif seg_start.day == 1:
                part = "P1"; period = f"{cursor.year:04d}-{cursor.month:02d}P1"
            else:
                part = "P2"; period = f"{cursor.year:04d}-{cursor.month:02d}P2"
            nb_days = (seg_end - seg_start).days + 1
            coef = round(nb_days / days_month, 8)
            rows.append({
                "FICHIER": row.get("FICHIER"),
                "NUMERO_DOMICILIATION": row.get("NUMERO_DOM_RETENU"),
                "DATE_DOMICILIATION": row.get("DATE_DOM_RETENUE"),
                "DATE_DOMICILIATION_SOURCE": "FICHIER_DOM" if row.get("DATE_DOM_RETENUE") else None,
                "NUMERO_CLIENT": row.get("NUMERO_CLIENT_RETENU"),
                "NOM_CLIENT": row.get("NOM_CLIENT_REFERENCE"),
                "NUMERO_CONTRAT": row.get("NUMERO_CONTRAT_REFERENCE"),
                "DATE_DEBUT_CONTRAT": start.isoformat(),
                "DATE_FIN_CONTRAT": end.isoformat(),
                "NUMERO_PERMIS_TRAVAIL": row.get("NUMERO_PERMIS_REFERENCE"),
                "PERIODE_TL": period, "MOIS_BASE": f"{cursor.year:04d}-{cursor.month:02d}",
                "PARTIE": part, "DATE_DEBUT_SEGMENT": seg_start.isoformat(),
                "DATE_FIN_SEGMENT": seg_end.isoformat(),
                "NB_JOURS_SEGMENT": round(float(nb_days), 2), "NB_JOURS_MOIS": round(float(days_month), 2),
                "COEFFICIENT_PRORATA": round(float(coef), 2),
                "SALAIRE_NET_REFERENCE": row.get("DOM_SALAIRE_NET_MENSUEL"),
                "TAUX_TRANSFERABLE_REFERENCE_RAW": row.get("DOM_TAUX_TRANSFERABLE"),
                "PLAFOND_MENSUEL_REFERENCE": plafond,
                "MONTANT_MAX_THEORIQUE": round(plafond * coef, 2) if plafond is not None else None,
                "MONTANT_AUTORISE_SAISI": None, "MONTANT_TRANSFERE": None, "SOLDE_RESTANT": None,
                "MATCH_STATUS": row.get("MATCH_STATUS"), "MATCH_METHOD": row.get("MATCH_METHOD"),
                "MATCH_SCORE": row.get("MATCH_SCORE"),
                "MATCH_CANDIDATES_COUNT": row.get("MATCH_CANDIDATES_COUNT"),
                "REFERENCE_PREDOM": row.get("REFERENCE_PREDOM_RETENUE"),
                "SALAIRE_ANCIEN": row.get("SALAIRE_ANCIEN"),
                "SALAIRE_NOUVEAU_AUGMENTE": row.get("SALAIRE_NOUVEAU_AUGMENTE"),
                "AUGMENTATION_DETECTEE": row.get("AUGMENTATION_DETECTEE"),
                "MONTANT_AUGMENTATION": row.get("MONTANT_AUGMENTATION"),
                "TAUX_AUGMENTATION_PCT": row.get("TAUX_AUGMENTATION_PCT"),
                "ALERTE_DOM_SUR_ANCIEN_SALAIRE": row.get("ALERTE_DOM_SUR_ANCIEN_SALAIRE"),
            })
        cursor = (date(cursor.year + 1, 1, 1) if cursor.month == 12
                  else date(cursor.year, cursor.month + 1, 1))
    return rows

print("✅ Consolidation + matching + planning OK")


In [ ]:
# =====================================================================
# 11. Checkpoints (un JSON canonique par PDF) + export Excel
# =====================================================================
def canonical_checkpoint_path(pdf_path):
    return JSON_DIR / f"{pdf_path.stem}.json"

def checkpoint_is_complete(dossier, pdf_path):
    if not isinstance(dossier, dict):
        return False
    stats = dossier.get("stats") or {}
    if dossier.get("source_file") != pdf_path.name:
        return False
    if int(stats.get("pages", 0) or 0) <= 0 or not dossier.get("page_records"):
        return False
    stored_hash = dossier.get("source_sha256")
    if not stored_hash:
        return False
    try:
        return stored_hash == sha256_file(pdf_path)
    except Exception:
        return False

def load_existing_checkpoint(pdf_path):
    canonical = canonical_checkpoint_path(pdf_path)
    if not canonical.exists():
        return None
    try:
        dossier = json.loads(canonical.read_text(encoding="utf-8"))
    except Exception:
        return None
    return dossier if checkpoint_is_complete(dossier, pdf_path) else None

def enrich_dossier_row_with_stats(dossier, statut):
    row = dossier.get("dossier_row") or {}
    stats = dossier.get("stats") or {}
    row.update({
        "STATUT_TRAITEMENT_PIPELINE": statut,
        "TEMPS_ECOULE_DOSSIER_S": round(float(stats.get("elapsed_s", 0) or 0), 2),
        "TOKENS_IN_DOSSIER": int(stats.get("tokens_in", 0) or 0),
        "TOKENS_OUT_DOSSIER": int(stats.get("tokens_out", 0) or 0),
        "TOKENS_TOTAL_DOSSIER": int(stats.get("tokens_total", 0) or 0),
    })
    dossier["dossier_row"] = row
    return dossier

def format_duration(seconds):
    seconds = max(0, int(round(float(seconds or 0))))
    hours, rem = divmod(seconds, 3600)
    minutes, secs = divmod(rem, 60)
    return f"{hours:02d}:{minutes:02d}:{secs:02d}" if hours else f"{minutes:02d}:{secs:02d}"

# --- Export Excel ------------------------------------------------------
COLONNES_MONTANT_V8 = {
    "SALAIRE_ANCIEN", "SALAIRE_NOUVEAU_AUGMENTE",
    "MONTANT_AUGMENTATION", "ECART_SALAIRE_DOM_CONTRAT",
}

def ordered_columns(rows):
    preferred = [
        "FICHIER", "NB_PAGES", "TYPES_DOCUMENTS",
        "STATUT_TRAITEMENT_PIPELINE", "TEMPS_ECOULE_DOSSIER_S",
        "TOKENS_IN_DOSSIER", "TOKENS_OUT_DOSSIER", "TOKENS_TOTAL_DOSSIER",
        "REFERENCE_DOM_EXTRAITE_RAW", "REFERENCE_DOM_EXTRAITE_NORMALISEE",
        "NUMERO_DOM_RETENU", "DATE_DOM_RETENUE", "NUMERO_CLIENT_RETENU",
        "NOM_CLIENT_REFERENCE", "NUMERO_CONTRAT_REFERENCE",
        "DATE_DEBUT_CONTRAT_REFERENCE", "DATE_FIN_CONTRAT_REFERENCE",
        "NUMERO_PERMIS_REFERENCE", "MATCH_SOURCE", "MATCH_METHOD",
        "MATCH_SCORE", "MATCH_STATUS", "MATCH_CANDIDATES_COUNT",
        "REFERENCE_PREDOM_RETENUE", "DATE_DEBUT_CONTRAT_PREDOM",
        "DATE_FIN_CONTRAT_PREDOM", "PREDOM_TROUVEE",
        "SALAIRE_ANCIEN", "SALAIRE_NOUVEAU_AUGMENTE",
        "AUGMENTATION_DETECTEE", "SENS_VARIATION_SALAIRE",
        "MONTANT_AUGMENTATION", "TAUX_AUGMENTATION_PCT",
        "SOURCE_AUGMENTATION", "CTS_LIGNE_SALAIRE_BRUTE",
        "ECART_SALAIRE_DOM_CONTRAT", "COHERENCE_SALAIRE_DOM_CONTRAT",
        "ALERTE_DOM_SUR_ANCIEN_SALAIRE",
        "NOM_TRAVAILLEUR_REFERENCE", "DATE_NAISSANCE_REFERENCE",
        "NATIONALITE_REFERENCE", "DATE_ENTREE_ALGERIE", "PERMIS_TRAVAIL_LU",
    ]
    cols = set().union(*(r.keys() for r in rows)) if rows else set()
    return [c for c in preferred if c in cols] + sorted(cols - set(preferred))

def sheet_from_rows(wb, title, rows, columns=None, amount_columns=None):
    ws = wb.create_sheet(title)
    columns = columns or ordered_columns(rows)
    if not columns:
        ws["A1"] = "Aucune donnée"
        return
    amount_columns = set(amount_columns or [])
    fill = PatternFill("solid", fgColor="1F4E78")
    font = Font(color="FFFFFF", bold=True, name="Arial", size=9)
    for j, name in enumerate(columns, 1):
        c = ws.cell(1, j, name); c.fill = fill; c.font = font
        c.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    for i, row in enumerate(rows, 2):
        for j, name in enumerate(columns, 1):
            value = row.get(name)
            if isinstance(value, (dict, list)):
                value = json.dumps(value, ensure_ascii=False)
            c = ws.cell(i, j, value)
            if name in amount_columns and isinstance(value, (int, float)):
                c.number_format = EXCEL_AMOUNT_FORMAT
    ws.freeze_panes = "A2"; ws.auto_filter.ref = ws.dimensions
    for j, name in enumerate(columns, 1):
        ws.column_dimensions[get_column_letter(j)].width = (
            38 if "ADRESSE" in name else min(34, max(14, len(name) + 2)))

def build_long_raw_rows(dossiers):
    rows = []
    for d in dossiers:
        for page in d.get("page_records", []):
            for field, value in (page.get("raw_data") or {}).items():
                rows.append({
                    "FICHIER": d.get("source_file"), "PAGE": page.get("page_num"),
                    "TYPE_DOCUMENT": page.get("doc_type"), "CHAMP": field,
                    "VALEUR_BRUTE": value,
                    "EXTRACTION_METHOD": page.get("extraction_method"),
                    "STATUT_EXTRACTION": page.get("extraction_status"),
                })
    return rows

def create_excel(excel_path, dossiers, errors, reference_df=None):
    pages, dossier_rows, planning, matching = [], [], [], []
    for d in dossiers:
        pages.extend(d.get("page_rows", []))
        row = d.get("dossier_row") or {}
        dossier_rows.append(row)
        planning.extend(month_segment_rows(row))
        matching.append({k: row.get(k) for k in [
            "FICHIER", "STATUT_TRAITEMENT_PIPELINE",
            "TEMPS_ECOULE_DOSSIER_S", "TOKENS_IN_DOSSIER",
            "TOKENS_OUT_DOSSIER", "TOKENS_TOTAL_DOSSIER",
            "NOM_CLIENT_REFERENCE", "NUMERO_CONTRAT_REFERENCE",
            "DATE_DEBUT_CONTRAT_REFERENCE", "DATE_FIN_CONTRAT_REFERENCE",
            "REFERENCE_DOM_EXTRAITE_RAW", "REFERENCE_DOM_EXTRAITE_NORMALISEE",
            "MATCH_SOURCE", "MATCH_METHOD", "MATCH_SCORE", "MATCH_STATUS",
            "MATCH_CANDIDATES_COUNT", "NUMERO_CLIENT_RETENU",
            "NUMERO_DOM_RETENU", "DATE_DOM_RETENUE",
            "REFERENCE_PREDOM_RETENUE", "DATE_DEBUT_CONTRAT_PREDOM",
            "DATE_FIN_CONTRAT_PREDOM", "PREDOM_TROUVEE",
            "AUGMENTATION_DETECTEE", "ALERTE_DOM_SUR_ANCIEN_SALAIRE",
        ]})
    raw = build_long_raw_rows(dossiers)
    suivi = []
    for d in dossiers:
        stats = d.get("stats") or {}
        row = d.get("dossier_row") or {}
        suivi.append({
            "FICHIER": d.get("source_file"),
            "STATUT_TRAITEMENT_PIPELINE": row.get("STATUT_TRAITEMENT_PIPELINE"),
            "NB_PAGES": stats.get("pages"),
            "TEMPS_ECOULE_DOSSIER_S": row.get("TEMPS_ECOULE_DOSSIER_S"),
            "TOKENS_IN_DOSSIER": row.get("TOKENS_IN_DOSSIER"),
            "TOKENS_OUT_DOSSIER": row.get("TOKENS_OUT_DOSSIER"),
            "TOKENS_TOTAL_DOSSIER": row.get("TOKENS_TOTAL_DOSSIER"),
            "PIPELINE_VERSION_JSON": d.get("pipeline_version"),
            "SHA256": d.get("source_sha256"),
        })
    wb = Workbook(); wb.remove(wb.active)
    sheet_from_rows(wb, "DOSSIERS_DOMICILIATION", dossier_rows,
                    amount_columns=AMOUNT_FIELDS | COLONNES_MONTANT_V8)
    sheet_from_rows(wb, "SUIVI_TRAITEMENT", suivi, [
        "FICHIER", "STATUT_TRAITEMENT_PIPELINE", "NB_PAGES",
        "TEMPS_ECOULE_DOSSIER_S", "TOKENS_IN_DOSSIER",
        "TOKENS_OUT_DOSSIER", "TOKENS_TOTAL_DOSSIER",
        "PIPELINE_VERSION_JSON", "SHA256",
    ], amount_columns={"TEMPS_ECOULE_DOSSIER_S"})
    sheet_from_rows(wb, "PLANNING_TL", planning, amount_columns={
        "SALAIRE_NET_REFERENCE", "PLAFOND_MENSUEL_REFERENCE",
        "MONTANT_MAX_THEORIQUE", "MONTANT_AUTORISE_SAISI",
        "MONTANT_TRANSFERE", "SOLDE_RESTANT", "NB_JOURS_SEGMENT",
        "NB_JOURS_MOIS", "COEFFICIENT_PRORATA",
        "SALAIRE_ANCIEN", "SALAIRE_NOUVEAU_AUGMENTE", "MONTANT_AUGMENTATION",
    })
    sheet_from_rows(wb, "PAGES_DOCUMENTS", pages)
    sheet_from_rows(wb, "EXTRACTION_BRUTE", raw, [
        "FICHIER", "PAGE", "TYPE_DOCUMENT", "CHAMP", "VALEUR_BRUTE",
        "EXTRACTION_METHOD", "STATUT_EXTRACTION",
    ])
    sheet_from_rows(wb, "MATCHING_DOM", matching)
    sheet_from_rows(wb, "ERREURS", errors)
    if reference_df is not None:
        sheet_from_rows(wb, "REFERENTIEL_DOM_PREDOM",
                        reference_df.where(pd.notna(reference_df), None).to_dict("records"))
    wb.save(excel_path)
    print(f"✅ Excel créé : {excel_path} | dossiers={len(dossier_rows)} | planning={len(planning)}")

print("✅ Checkpoints + export Excel OK")


In [ ]:
# =====================================================================
# 12. Passe 1 (moteur léger) : rendu, classification, extraction
# =====================================================================
# Construit les records de chaque PDF. Alimente deux files pour la
# passe lourde : pending_permis (planches) et pending_escalades (dactylo
# sous-rempli par le 7B). Les images ne sont conservées en mémoire que
# pour ces pages en attente.

pending_permis = []     # dicts {pdf_path, page_index, record, crops, doc_type}
pending_escalades = []  # records dactylo (champ _pdf_path ajouté)

def _blank_record(p):
    return {
        "page_num": p["page_num"],
        "doc_type": "AUTRE", "classification_method": None,
        "classification_confidence": None, "bloc_identite_present": None,
        "classification_tokens_in": 0, "classification_tokens_out": 0,
        "raw_data": {}, "extraction_method": None, "extraction_status": "NON_LANCEE",
        "extraction_taux_remplissage": 0.0, "extraction_strategies": [],
        "extraction_tokens_in": 0, "extraction_tokens_out": 0,
    }

def process_pdf_pass1(pdf_path):
    t0 = time.time()
    pages = pdf_to_pages(pdf_path)
    records = []
    scanned_pages = []

    with fitz.open(str(pdf_path)) as doc:
        texts = [(doc.load_page(p["index"]).get_text("text") or "") for p in pages]

    for p, txt in zip(pages, texts):
        if is_blank(p["image"]):
            rec = _blank_record(p)
            rec["extraction_status"] = "PAGE_BLANCHE"
            records.append(rec)
            continue
        kw = native_keyword_classify(txt) if len(txt.strip()) > 200 else None
        if kw:
            raw = extract_native(txt, kw)
            champs = CHAMPS_ATTENDUS.get(kw) or []
            for c in champs:
                raw.setdefault(c, None)
            rec = _blank_record(p)
            rec.update({
                "doc_type": kw, "classification_method": "MOTS_CLES_TEXTE_NATIF",
                "classification_confidence": 1.0, "raw_data": raw,
                "extraction_method": "REGEX_TEXTE_NATIF",
                "extraction_taux_remplissage": taux_remplissage(raw, champs),
            })
            rec["extraction_status"] = ("OK" if rec["extraction_taux_remplissage"] >= SEUIL_REMPLISSAGE_OK
                                        else "PARTIELLE")
            records.append(rec)
        elif len(txt.strip()) > 200:
            # PDF né numérique mais type non reconnu : ne pas brûler de tokens.
            rec = _blank_record(p)
            rec.update({"classification_method": "TEXTE_NATIF_SANS_TYPE",
                        "extraction_status": "NON_APPLICABLE"})
            records.append(rec)
        else:
            rec = _blank_record(p)
            rec["_page_ref"] = p           # image conservée pour la suite
            records.append(rec)
            scanned_pages.append(rec)

    # Classification VLM groupée des pages scannées
    if scanned_pages:
        vlm_classify_pages(scanned_pages)

    # Extraction VLM groupée par type (moteur léger)
    for doc_type in TYPES_DACTYLO:
        group = [r for r in scanned_pages if r["doc_type"] == doc_type]
        for r in group:
            r["_pdf_path"] = str(pdf_path)
        escalades = vlm_extract_grouped(group, doc_type)
        pending_escalades.extend(escalades)

    # Planches permis -> passe lourde (le 7B reste sur les dactylo)
    for r in scanned_pages:
        if r["doc_type"] in TYPES_PERMIS:
            p = r["_page_ref"]
            r["_pdf_path"] = str(pdf_path)
            r["extraction_status"] = "EN_ATTENTE_PASSE_LOURDE"
            pending_permis.append({
                "pdf_path": str(pdf_path),
                "page_index": p["index"],
                "record": r,
                "crops": crops_planche_permis(p["image"]),
                "doc_type": r["doc_type"],
            })
        if r["extraction_status"] == "NON_LANCEE" and r["doc_type"] == "AUTRE":
            r["extraction_status"] = "NON_APPLICABLE"

    # Libération des images devenues inutiles (sauf escalades en attente)
    escalade_ids = {id(r) for r in pending_escalades}
    for r in records:
        if "_page_ref" in r and id(r) not in escalade_ids:
            del r["_page_ref"]

    tokens_in = sum(r.get("classification_tokens_in", 0) + r.get("extraction_tokens_in", 0) for r in records)
    tokens_out = sum(r.get("classification_tokens_out", 0) + r.get("extraction_tokens_out", 0) for r in records)
    return {
        "source_file": pdf_path.name,
        "source_sha256": sha256_file(pdf_path),
        "pipeline_version": PIPELINE_VERSION,
        "records": records,
        "elapsed_s": round(time.time() - t0, 3),
        "tokens_in": tokens_in,
        "tokens_out": tokens_out,
    }

print("✅ Passe 1 OK (prête à être lancée par la cellule d'exécution)")


In [ ]:
# =====================================================================
# 13. Tests techniques (sans GPU)
# =====================================================================
# Ligne de salaire
_cas = split_ligne_salaire("Salaire mensuel de base net :  506,471.38  au lieu de  479,274.29")
assert _cas["nouveau"] == 506471.38 and _cas["ancien"] == 479274.29 and _cas["mention_presente"]
_cas_fr = split_ligne_salaire("Salaire mensuel de base net : 506 471,38 au lieu de 479 274,29")
assert _cas_fr["nouveau"] == 506471.38 and _cas_fr["ancien"] == 479274.29
assert split_ligne_salaire("Salaire mensuel de base net : 466,300.88")["ancien"] is None
assert split_ligne_salaire(None)["ancien"] is None

# Référence DOM
assert normalize_dom_reference("271901|2026.1|40|00119|DZD") == "271901202614000119DZD"
assert normalize_dom_reference("2026.1.40-00119") == "271901202614000119DZD"
assert dom_reference_is_valid("271901202614000119DZD")

# Planning P1/P2
_demo = {
    "FICHIER": "demo.pdf", "DATE_DEBUT_CONTRAT_REFERENCE": "12/06/2025",
    "DATE_FIN_CONTRAT_REFERENCE": "11/06/2026", "DOM_PART_TRANSFERABLE": 442985.84,
    "DOM_SALAIRE_NET_MENSUEL": 466300.88, "DOM_TAUX_TRANSFERABLE": "95%",
    "NUMERO_CONTRAT_REFERENCE": "DEMO", "NOM_CLIENT_REFERENCE": "CLIENT DEMO",
    "NUMERO_PERMIS_REFERENCE": "PERMIS-DEMO", "NUMERO_DOM_RETENU": "DOM-DEMO",
    "DATE_DOM_RETENUE": "01/01/2025", "NUMERO_CLIENT_RETENU": "CLIENT-001",
    "MATCH_STATUS": "TEST",
}
_demo_rows = month_segment_rows(_demo)
assert len(_demo_rows) == 13
assert _demo_rows[0]["PERIODE_TL"] == "2025-06P2"
assert _demo_rows[-1]["PERIODE_TL"] == "2026-06P1"
assert _demo_rows[0]["NB_JOURS_SEGMENT"] == 19 and _demo_rows[-1]["NB_JOURS_SEGMENT"] == 11
_full = dict(_demo, DATE_DEBUT_CONTRAT_REFERENCE="01/05/2026", DATE_FIN_CONTRAT_REFERENCE="31/05/2026")
_full_rows = month_segment_rows(_full)
assert len(_full_rows) == 1 and _full_rows[0]["PERIODE_TL"] == "2026-05" and _full_rows[0]["PARTIE"] is None

# Consolidation augmentation
_rec_aug = [
    {"page_num": 1, "doc_type": "CONTRAT_SPECIFIQUE", "raw_data": {
        "CTS_SALAIRE_NET": "506471.38", "CTS_SALAIRE_NET_ANCIEN": "479274.29",
        "CTS_LIGNE_SALAIRE_BRUTE": "Salaire mensuel de base net : 506,471.38 au lieu de 479,274.29",
        "CTS_NOM_PRENOM_TRAVAILLEUR": "YILDIRIM IBRAHIM"}},
    {"page_num": 2, "doc_type": "ENGAGEMENT_DOMICILIATION", "raw_data": {
        "DOM_SALAIRE_NET_MENSUEL": "506471,38", "DOM_PART_TRANSFERABLE": "481147.81"}},
]
_row = consolidate_dossier("test.pdf", _rec_aug)
assert _row["AUGMENTATION_DETECTEE"] is True
assert _row["MONTANT_AUGMENTATION"] == 27197.09
assert _row["TAUX_AUGMENTATION_PCT"] == 5.67
assert _row["COHERENCE_SALAIRE_DOM_CONTRAT"] is True
assert _row["ALERTE_DOM_SUR_ANCIEN_SALAIRE"] is False

# Frontière de planche
_planche = Image.new("L", (800, 2000), 255)
_px = _planche.load()
for y in list(range(60, 880)) + list(range(1120, 1940)):
    for x in range(40, 760, 3):
        _px[x, y] = 0
_frontiere = detecter_frontiere_documents(_planche.convert("RGB"))
assert 0.44 <= _frontiere <= 0.52, _frontiere
assert detecter_frontiere_documents(Image.new("RGB", (800, 1200), "white")) == FRONTIERE_DEFAUT

# Mots-clés natifs
assert native_keyword_classify("CONTRAT DE TRAVAIL SPECIFIQUE A LA MAIN D'OEUVRE ETRANGERE") == "CONTRAT_SPECIFIQUE"
assert native_keyword_classify("CONTRAT DE TRAVAIL A DUREE DETERMINEE") == "CONTRAT_TRAVAIL"
assert native_keyword_classify("document sans titre connu") is None
assert native_keyword_classify("ENGAGEMENT DE DOMICILIATION CONTRAT DES SALARIES ETRANGERS") == "ENGAGEMENT_DOMICILIATION"

print("✅ Tous les tests V8 sont passés")


In [ ]:
# =====================================================================
# 14. Exécution complète : passe 1 (7B) -> passe lourde (35B-A3B) -> export
# =====================================================================
if not pdfs:
    raise RuntimeError(f"Aucun PDF dans {INPUT_DIR}")

ram_free = psutil.virtual_memory().available / 1e9
log(f"RAM libre : {ram_free:.1f} GB | PDF : {len(pdfs)}")

# --- Contrôle d'intégrité rapide des PDFs ------------------------------
for p in pdfs:
    with fitz.open(str(p)) as doc:
        if int(doc.page_count) <= 0:
            raise RuntimeError(f"{p.name} : 0 page")
print(f"✅ Diagnostic PDF : {len(pdfs)} fichier(s)")

reference_df = build_reference_table()
log(f"Référentiel externe : {0 if reference_df is None else len(reference_df)} ligne(s)")

all_dossiers, errors = [], []
nb_repris = nb_nouveaux = 0
partial = {}          # pdf_name -> pass1 result (en attente de passe lourde)
pipeline_start = time.time()

# ---------------------------------------------------------------------
# Passe 1 : moteur léger — classification + extraction dactylographiée
# ---------------------------------------------------------------------
load_engine_light()
position = 0
for pdf_path in pdfs:
    position += 1
    try:
        ckpt = load_existing_checkpoint(pdf_path)
        if ckpt:
            all_dossiers.append(enrich_dossier_row_with_stats(ckpt, "REPRIS_JSON_EXISTANT"))
            nb_repris += 1
            print(f"[{position}/{len(pdfs)}] {pdf_path.name} | SKIP (JSON existant)", flush=True)
            continue
        result = process_pdf_pass1(pdf_path)
        partial[pdf_path.name] = result
        nb_nouveaux += 1
        print(f"[{position}/{len(pdfs)}] {pdf_path.name} | passe 1 OK "
              f"({result['elapsed_s']:.1f}s | IN~{result['tokens_in']:,} | OUT~{result['tokens_out']:,} "
              f"| permis en attente : {sum(1 for it in pending_permis if it['pdf_path'] == str(pdf_path))})", flush=True)
    except Exception as exc:
        errors.append({"FICHIER": pdf_path.name, "ETAPE": "PASSE_1", "ERREUR": repr(exc),
                       "DATE": datetime.now().isoformat(timespec="seconds")})
        print(f"[{position}/{len(pdfs)}] {pdf_path.name} | ERREUR passe 1 : {exc!r}", flush=True)
        log(f"❌ {pdf_path.name} passe 1 : {exc}")

# ---------------------------------------------------------------------
# Bascule : libération du léger, chargement du lourd
# ---------------------------------------------------------------------
unload_engine("light")
load_engine_heavy()

# --- Passe lourde A : escalades dactylographiées ----------------------
if pending_escalades:
    log(f"Passe lourde : {len(pending_escalades)} page(s) dactylo en escalade")
    heavy_extract_escalades(pending_escalades)

# --- Passe lourde B : planches permis ---------------------------------
if pending_permis:
    log(f"Passe lourde : {len(pending_permis)} planche(s) permis")
    heavy_extract_permis(pending_permis)

# ---------------------------------------------------------------------
# Finalisation : consolidation, matching, checkpoints, export
# ---------------------------------------------------------------------
for pdf_path in pdfs:
    if pdf_path.name not in partial:
        continue
    try:
        result = partial[pdf_path.name]
        records = []
        for r in result["records"]:
            r = {k: v for k, v in r.items() if not k.startswith("_")}
            records.append(r)
        tokens_in = int(result["tokens_in"])
        tokens_out = int(result["tokens_out"])
        dossier = {
            "source_file": result["source_file"],
            "source_sha256": result["source_sha256"],
            "pipeline_version": PIPELINE_VERSION,
            "stats": {"pages": len(records), "tokens_in": tokens_in,
                      "tokens_out": tokens_out, "tokens_total": tokens_in + tokens_out,
                      "elapsed_s": result["elapsed_s"]},
            "page_records": records,
            "page_rows": [build_page_row(pdf_path.name, r) for r in records],
            "dossier_row": {},
            "planning_tl": [],
        }
        dossier["dossier_row"] = consolidate_dossier(pdf_path.name, records)
        dossier["dossier_row"].update({
            "STATUT_TRAITEMENT_PIPELINE": "TRAITE_NOUVEAU",
            "TEMPS_ECOULE_DOSSIER_S": dossier["stats"]["elapsed_s"],
            "TOKENS_IN_DOSSIER": tokens_in, "TOKENS_OUT_DOSSIER": tokens_out,
            "TOKENS_TOTAL_DOSSIER": tokens_in + tokens_out,
        })
        dossier["dossier_row"].update(match_dossier(dossier["dossier_row"], reference_df))
        canonical_checkpoint_path(pdf_path).write_text(
            json.dumps(dossier, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
        # Fusion avec les dossiers repris (un seul exemplaire par PDF)
        all_dossiers = [d for d in all_dossiers if d.get("source_file") != pdf_path.name]
        all_dossiers.append(dossier)
    except Exception as exc:
        errors.append({"FICHIER": pdf_path.name, "ETAPE": "FINALISATION", "ERREUR": repr(exc),
                       "DATE": datetime.now().isoformat(timespec="seconds")})
        log(f"❌ {pdf_path.name} finalisation : {exc}")

unload_engine("heavy")

create_excel(EXCEL_PATH, all_dossiers, errors, reference_df)
with open(MASTER_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump({"generated_at": datetime.now().isoformat(timespec="seconds"),
               "pipeline_version": PIPELINE_VERSION,
               "dossiers": all_dossiers, "errors": errors},
              f, ensure_ascii=False, indent=2, default=str)

elapsed_pipeline = time.time() - pipeline_start
total_in = sum(int((d.get("stats") or {}).get("tokens_in", 0) or 0) for d in all_dossiers)
total_out = sum(int((d.get("stats") or {}).get("tokens_out", 0) or 0) for d in all_dossiers)
print(f"\nTerminé | total={len(all_dossiers)} | traités={nb_nouveaux} | skip={nb_repris} "
      f"| erreurs={len(errors)} | IN~{total_in:,} | OUT~{total_out:,} "
      f"| durée={format_duration(elapsed_pipeline)}", flush=True)
log(f"✅ Pipeline V8 terminé | dossiers={len(all_dossiers)} | nouveaux={nb_nouveaux} "
    f"| repris={nb_repris} | erreurs={len(errors)}")


## 15. Lecture des résultats

- `domiciliations_master.json` : référentiel consolidé.
- `DOSSIERS_DOMICILIATION` : une ligne par contrat/domiciliation (avec les
  colonnes d'augmentation de salaire et d'identité V7.1/V8).
- `PLANNING_TL` : périodes de transfert (mois partiels en `P1`/`P2`).
- `PAGES_DOCUMENTS` : statut par page — regarder `CLASSIFICATION_METHOD`
  (`MOTS_CLES_TEXTE_NATIF`, `VLM_7B`), `EXTRACTION_METHOD`
  (`REGEX_TEXTE_NATIF`, `VLM_7B`, `VLM_35B_A3B`, `VLM_7B puis 35B_A3B`),
  `TAUX_REMPLISSAGE` et `STRATEGIES_UTILISEES`.
- `SUIVI_TRAITEMENT` : temps et tokens par dossier — à comparer avec la V7.1.
- `MATCHING_DOM`, `EXTRACTION_BRUTE`, `ERREURS`, `REFERENTIEL_DOM_PREDOM`.

### Points de vigilance V8

1. **Chemins de modèles** : adapter `MODELE_LEGER_PATH` / `MODELE_LOURD_PATH`
   au ModelHub Domino réel.
2. **Tokens** : `TOKENS_IN` est une estimation (image = ~1 token par bloc
   28×28, prompt = chars/3.5) ; les tokens de sortie sont exacts.
3. **Reprise sur incident** : si la passe 1 est interrompue, les PDF déjà
   finalisés sont repris via leur JSON canonique ; un PDF interrompu en passe
   1 est entièrement retraité (la passe 1 est rapide).
4. **Qualité** : la décision d'escalade du 7B vers le 35B-A3B est tracée dans
   `EXTRACTION_METHOD` — surveiller le volume d'escalades sur un échantillon
   pour valider le seuil 0.70.
5. **Réglementaire** : le pipeline ne fait pas les contrôles finaux ; les
   règles métier restent dans Alteryx. `ALERTE_DOM_SUR_ANCIEN_SALAIRE`
   signale une domiciliation restée sur un salaire périmé.
